# M3.S4 — Libraries, I/O & Reproducible Workflows
**HPC 2026 · Session 14 · Build `M3S4-2026-10-06-v9`**

**Predict → run → verify → change one setting → rerun → explain.**

A matrix calibration, rod heat balance, sensor signal, heat simulation and weather dataset connect the lesson to real scientific tasks. Real timings come from your machine, never from invented results. Each experiment reports **PASS**, **FAIL**, or **SKIPPED with a reason**. An unavailable optional package is not a successful demonstration.

Run the setup, edit settings, materialize sources, submit, then collect. Return to the numbered sections at the corresponding slides.

| Slides | Notebook | In-class activity |
|---|---|---|
| 2–9 | §1 | Predict frequency peaks and DFT/FFT growth |
| Library extension | §2A–2B | BLAS versus manual C; LAPACK heat solve and pivoting |
| 10–14 | §2 | Inspect active modules and the library actually used |
| 15–18 | §2D | Kill a heat simulation and resume its saved state |
| 19–23 | §3/§4 | Predict I/O fraction and compare equal payloads |
| 24–25 | §5–§7 | Identify storage; read a weather slice; MPI byte ownership |
| 26–28 | §8–§10 | Replay a saved manifest and reject altered inputs |
| 29–32 | §11–§13 | Separate environment, container and workflow responsibilities |

**Requirements:** Python + NumPy; GCC + FFTW3 for the compiled transform comparison; h5py or the HDF5 C wrapper for structured data; mpicc and an MPI launcher for parallel I/O. No automatic package installation. The runner and C sources are embedded in this notebook, so a downloaded notebook needs no further source downloads.


## 0 — How to use this learning notebook
**Start with a question:** what changes when we replace manual calculations with a library, save progress, or change how we write files?

1. Run **setup → materialize → submit → collect**. These cells run the real experiments in one Slurm job. The collection cell waits briefly; if queued, rerun only collection.
2. The numbered activity cells below **display the measured results**. They do not each start a new simulation. PASS means the stated numerical/data checks passed.
3. At each activity, read **what actually runs**, inspect the table/plot, and answer the question before looking at the outcome.
4. Change **one named setting in SETTINGS**, predict the result, then rerun setup/materialize/submit/collect and the activity. Record both run IDs. The unchanged settings/source reuse the previous run; `NEW_RUN=True` forces another repetition.

On SciTech keep `RUN_LOCAL=False`. Slurm requests one node, four MPI task slots by default, `blas_threads` CPUs per task, 4 GB and 10 minutes. Use a user-owned shared WORKSPACE. The scientific examples use NumPy; plots are optional. A laptop can use `RUN_LOCAL=True`; dependencies unavailable there are explicitly SKIPPED.

**What you should be able to explain:** where time goes, which work is parallel, why a faster library still needs correctness checks, what must be saved for restart, and what evidence another person needs to repeat a result.

**Reading the code:** each activity now shows a short excerpt from the actual worker or compiled C source. These excerpts omit setup and are not standalone cells. The scientific statements run in the batch job; the result cells display its measured output. Expand the complete source when you want the full runnable context.


In [ ]:
from pathlib import Path
import os, sys, json, time, shutil, subprocess, hashlib, uuid, statistics
BUILD = 'M3S4-2026-10-06-v9'
WORKSPACE = Path.cwd() / 'm3s4_runs'  # edit to a user-owned SHARED path visible to compute nodes
WORKSPACE.mkdir(parents=True, exist_ok=True)
RUN_LOCAL = False
NEW_RUN = False
WAIT_SECONDS = 45
PARTITION = 'cpu'
SETTINGS = {
    'blas_sizes': [128, 256, 512], 'blas_threads': 1,
    'lapack_points': 128, 'rod_left': 80.0, 'rod_right': 20.0, 'rod_source': 10.0,
    'sizes': [128, 256, 512, 1024], 'repeats': 5, 'fft_batch': 500, 'seed': 2026, 'input_seed': 12345,
    'files': 200, 'bytes_per_file': 4096, 'fsync': False,
    'heat_points': 128, 'heat_steps': 100, 'checkpoint_interval': 10, 'failure_step': 57,
    'noise_sigma': 0.01, 'input_elements': 50000,
    'mpi_ranks': 4, 'mpi_block': 1024,
    'weather_shape': [4, 32, 32], 'weather_slice': 2, 'compression': None,
}
print('Build:', BUILD, '| Slurm:', shutil.which('sbatch') or 'unavailable')
print('Workspace:', WORKSPACE.resolve())
print('Kernel:', sys.executable)


### Materialize the experiment sources
The following cell contains the actual implementations. Expand it to inspect the C DFT, MPI calls or checkpoint state. It writes them into the workspace; it does not install anything.

The serial transform comparison calls both compiled implementations through Python bindings. This separates the algorithm lesson from Python-versus-C interpreter overhead. FFT plan creation is measured separately; transform execution reuses the plan. FFTW and the C reference are serial here: **this is library reuse, not an OpenMP experiment.**


In [ ]:
SOURCES = {}
SOURCES['m3s4_dft.c'] = r'''/* Teaching reference: direct O(N^2) forward, unnormalized DFT of real data. */
#include <math.h>
void direct_dft(int n, const double *in, double *out) {
    const double tau = 2.0 * acos(-1.0);
    for (int k = 0; k < n; ++k) {
        double re = 0.0, im = 0.0;
        for (int j = 0; j < n; ++j) {
            double angle = -tau * (double)k * j / n;
            re += in[j] * cos(angle); im += in[j] * sin(angle);
        }
        out[2*k] = re; out[2*k+1] = im;
    }
}
'''
SOURCES['m3s4_linalg.c'] = r'''/* Teaching references: row-major dense products and Gaussian elimination.
 * Same double precision and compiler flags for both manual products. */
#include <math.h>
#include <string.h>
void matmul_ijk(int n, const double *a, const double *b, double *c) {
    for(int i=0;i<n;i++) for(int j=0;j<n;j++) {
        double sum=0;
        for(int k=0;k<n;k++) sum+=a[i*n+k]*b[k*n+j];
        c[i*n+j]=sum;
    }
}
void matmul_ikj(int n, const double *a, const double *b, double *c) {
    memset(c,0,(size_t)n*n*sizeof(double));
    for(int i=0;i<n;i++) for(int k=0;k<n;k++) {
        double v=a[i*n+k];
        for(int j=0;j<n;j++) c[i*n+j]+=v*b[k*n+j];
    }
}
/* Overwrites A and b, just like DGESV. A is row-major here.
 * Set pivot=0 to demonstrate why a nonzero leading pivot matters. */
int gaussian(int n, double *a, double *b, int pivot) {
    for(int k=0;k<n;k++) {
        int p=k;
        if(pivot) for(int i=k+1;i<n;i++) if(fabs(a[i*n+k])>fabs(a[p*n+k])) p=i;
        if(fabs(a[p*n+k])<1e-14) return k+1;
        if(p!=k) {
            for(int j=0;j<n;j++) {double t=a[k*n+j];a[k*n+j]=a[p*n+j];a[p*n+j]=t;}
            double t=b[k];b[k]=b[p];b[p]=t;
        }
        for(int i=k+1;i<n;i++) {
            double f=a[i*n+k]/a[k*n+k];a[i*n+k]=0;
            for(int j=k+1;j<n;j++) a[i*n+j]-=f*a[k*n+j];
            b[i]-=f*b[k];
        }
    }
    for(int i=n-1;i>=0;i--) {
        double v=b[i];
        for(int j=i+1;j<n;j++) v-=a[i*n+j]*b[j];
        b[i]=v/a[i*n+i];
    }
    return 0;
}
'''
SOURCES['m3s4_mpiio.c'] = r'''/* Every rank owns a disjoint byte interval. Verify EVERY byte after reopen. */
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#define CHECK(call) do { int rc=(call); if(rc!=MPI_SUCCESS) { char msg[MPI_MAX_ERROR_STRING]; int n=0; MPI_Error_string(rc,msg,&n); fprintf(stderr,"MPI error at %s:%d in %s: %.*s\\n",__FILE__,__LINE__,#call,n,msg); MPI_Abort(MPI_COMM_WORLD,rc); } } while(0)
int main(int argc, char **argv) {
    CHECK(MPI_Init(&argc,&argv));
    int rank,size; CHECK(MPI_Comm_rank(MPI_COMM_WORLD,&rank)); CHECK(MPI_Comm_size(MPI_COMM_WORLD,&size));
    int block=argc>1?atoi(argv[1]):1024;
    if(block<1 || block>1048576) MPI_Abort(MPI_COMM_WORLD,2);
    unsigned char *buf=malloc((size_t)block); if(!buf) MPI_Abort(MPI_COMM_WORLD,3);
    for(int j=0;j<block;j++) buf[j]=(unsigned char)((rank*17+j)%251);
    MPI_File fh; MPI_Status status;
    CHECK(MPI_File_open(MPI_COMM_WORLD,"mpi_output.bin",MPI_MODE_CREATE|MPI_MODE_RDWR,MPI_INFO_NULL,&fh));
    CHECK(MPI_File_set_size(fh,(MPI_Offset)size*block));
    CHECK(MPI_Barrier(MPI_COMM_WORLD)); double start=MPI_Wtime();
    /* Default file view has byte etypes, so this offset is in bytes. */
    CHECK(MPI_File_write_at_all(fh,(MPI_Offset)rank*block,buf,block,MPI_BYTE,&status));
    int count; CHECK(MPI_Get_count(&status,MPI_BYTE,&count)); if(count!=block) MPI_Abort(MPI_COMM_WORLD,4);
    CHECK(MPI_File_sync(fh)); CHECK(MPI_File_close(&fh));
    double elapsed=MPI_Wtime()-start,maximum; CHECK(MPI_Reduce(&elapsed,&maximum,1,MPI_DOUBLE,MPI_MAX,0,MPI_COMM_WORLD));
    CHECK(MPI_File_open(MPI_COMM_WORLD,"mpi_output.bin",MPI_MODE_RDONLY,MPI_INFO_NULL,&fh));
    MPI_Offset bytes; CHECK(MPI_File_get_size(fh,&bytes));
    memset(buf,0,(size_t)block);
    CHECK(MPI_File_read_at_all(fh,(MPI_Offset)rank*block,buf,block,MPI_BYTE,&status));
    CHECK(MPI_Get_count(&status,MPI_BYTE,&count));
    int ok=(bytes==(MPI_Offset)size*block && count==block);
    for(int j=0;j<block;j++) if(buf[j]!=(unsigned char)((rank*17+j)%251)) ok=0;
    CHECK(MPI_File_close(&fh)); int all_ok; CHECK(MPI_Allreduce(&ok,&all_ok,1,MPI_INT,MPI_MIN,MPI_COMM_WORLD));
    if(rank==0) printf("{\"passed\":%s,\"ranks\":%d,\"block\":%d,\"bytes\":%lld,\"max_write_sync_close_s\":%.9f}\n",all_ok?"true":"false",size,block,(long long)bytes,maximum);
    free(buf); CHECK(MPI_Finalize()); return all_ok?0:5;
}
'''
SOURCES['m3s4_hdf5.c'] = r'''/* Weather field, editable dimensions, chunking/compression and hyperslab read.
 * A serial writer even when linked with a parallel HDF5 installation. */
#include <hdf5.h>
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#define HANDLE(v) do { if((v)<0) return 2; } while(0)
#define CALL(c) do { if((c)<0) return 3; } while(0)
static double now(void) { struct timespec t; clock_gettime(CLOCK_MONOTONIC,&t);return t.tv_sec+t.tv_nsec*1e-9; }
static int attribute(hid_t d,const char *name,const char *value) {
    hid_t s=H5Screate(H5S_SCALAR),t=H5Tcopy(H5T_C_S1);HANDLE(s);HANDLE(t);
    CALL(H5Tset_size(t,strlen(value)+1));
    hid_t a=H5Acreate2(d,name,t,s,H5P_DEFAULT,H5P_DEFAULT);HANDLE(a);CALL(H5Awrite(a,t,value));
    CALL(H5Aclose(a));CALL(H5Tclose(t));CALL(H5Sclose(s));return 0;
}
int main(int argc,char **argv) {
    if(argc!=6) return 1;
    int nt=atoi(argv[1]),ny=atoi(argv[2]),nx=atoi(argv[3]),selected=atoi(argv[4]),zip=atoi(argv[5]);
    if(nt<1||ny<1||nx<1||selected<0||selected>=nt||(zip!=0&&zip!=1)) return 1;
    size_t plane=(size_t)ny*nx,total=(size_t)nt*plane;
    if(total>2000000) return 1;
    double *field=malloc(total*sizeof(double)),*readback=malloc(total*sizeof(double)),*slice=malloc(plane*sizeof(double)),*times=malloc((size_t)nt*sizeof(double));
    if(!field||!readback||!slice||!times) return 1;
    for(int t=0;t<nt;t++) {times[t]=t;for(int y=0;y<ny;y++) for(int x=0;x<nx;x++) field[(size_t)t*plane+(size_t)y*nx+x]=273.15+t+0.1*y+0.01*x;}
    hsize_t dims[3]={(hsize_t)nt,(hsize_t)ny,(hsize_t)nx},chunk[3]={1,(hsize_t)ny,(hsize_t)nx};
    double start=now();
    hid_t f=H5Fcreate("weather.h5",H5F_ACC_TRUNC,H5P_DEFAULT,H5P_DEFAULT);HANDLE(f);
    hid_t s=H5Screate_simple(3,dims,NULL),props=H5Pcreate(H5P_DATASET_CREATE);HANDLE(s);HANDLE(props);
    CALL(H5Pset_chunk(props,3,chunk));
    if(zip) CALL(H5Pset_deflate(props,4));
    hid_t d=H5Dcreate2(f,"temperature",H5T_IEEE_F64LE,s,H5P_DEFAULT,props,H5P_DEFAULT);HANDLE(d);
    CALL(H5Dwrite(d,H5T_NATIVE_DOUBLE,H5S_ALL,H5S_ALL,H5P_DEFAULT,field));
    if(attribute(d,"units","K")||attribute(d,"dimensions","time,y,x")) return 3;
    CALL(H5Dclose(d));CALL(H5Pclose(props));CALL(H5Sclose(s));
    hsize_t time_dim[1]={(hsize_t)nt};s=H5Screate_simple(1,time_dim,NULL);HANDLE(s);
    d=H5Dcreate2(f,"time",H5T_IEEE_F64LE,s,H5P_DEFAULT,H5P_DEFAULT,H5P_DEFAULT);HANDLE(d);
    CALL(H5Dwrite(d,H5T_NATIVE_DOUBLE,H5S_ALL,H5S_ALL,H5P_DEFAULT,times));
    if(attribute(d,"units","hours since simulation start")) return 3;
    CALL(H5Dclose(d));CALL(H5Sclose(s));CALL(H5Fclose(f));double write_s=now()-start;
    f=H5Fopen("weather.h5",H5F_ACC_RDONLY,H5P_DEFAULT);HANDLE(f);
    d=H5Dopen2(f,"temperature",H5P_DEFAULT);HANDLE(d);s=H5Dget_space(d);HANDLE(s);
    hsize_t got[3];if(H5Sget_simple_extent_ndims(s)!=3) return 4;CALL(H5Sget_simple_extent_dims(s,got,NULL));
    for(int j=0;j<3;j++) if(got[j]!=dims[j]) return 5;
    CALL(H5Dread(d,H5T_NATIVE_DOUBLE,H5S_ALL,H5S_ALL,H5P_DEFAULT,readback));
    for(size_t i=0;i<total;i++) if(fabs(readback[i]-field[i])>1e-12) return 6;
    hid_t a=H5Aopen(d,"units",H5P_DEFAULT);HANDLE(a);hid_t type=H5Aget_type(a);HANDLE(type);
    if(H5Tget_size(type)!=2) return 7;
    char units[2];CALL(H5Aread(a,type,units));if(units[0]!='K'||units[1]!=0) return 8;
    CALL(H5Aclose(a));CALL(H5Tclose(type));
    hsize_t offset[3]={(hsize_t)selected,0,0},count[3]={1,(hsize_t)ny,(hsize_t)nx},md[2]={(hsize_t)ny,(hsize_t)nx};
    CALL(H5Sselect_hyperslab(s,H5S_SELECT_SET,offset,NULL,count,NULL));
    hid_t memory=H5Screate_simple(2,md,NULL);HANDLE(memory);start=now();
    CALL(H5Dread(d,H5T_NATIVE_DOUBLE,memory,s,H5P_DEFAULT,slice));double read_s=now()-start;
    for(size_t i=0;i<plane;i++) if(fabs(slice[i]-field[(size_t)selected*plane+i])>1e-12) return 9;
    CALL(H5Sclose(memory));CALL(H5Sclose(s));CALL(H5Dclose(d));
    d=H5Dopen2(f,"time",H5P_DEFAULT);HANDLE(d);CALL(H5Dread(d,H5T_NATIVE_DOUBLE,H5S_ALL,H5S_ALL,H5P_DEFAULT,times));
    for(int t=0;t<nt;t++) if(times[t]!=t) return 10;
    CALL(H5Dclose(d));CALL(H5Fclose(f));
    unsigned major,minor,release;CALL(H5get_libversion(&major,&minor,&release));
    printf("{\"passed\":true,\"shape\":[%d,%d,%d],\"chunks\":[1,%d,%d],\"units\":\"K\",\"checked_values\":%zu,\"slice_time\":%d,\"compression\":%s,\"write_close_s\":%.9g,\"read_slice_s\":%.9g,\"hdf5\":\"%u.%u.%u\"}\n",nt,ny,nx,ny,nx,total,selected,zip?"\"gzip\"":"null",write_s,read_s,major,minor,release);
    free(field);free(readback);free(slice);free(times);return 0;
}
'''
SOURCES['m3s4_lab.py'] = r'''"""M3S4 measured exercises. All files stay inside a unique, user-owned run directory."""
from pathlib import Path
import argparse, ctypes as ct, ctypes.util, hashlib, json, math, os, platform
import shlex, shutil, statistics, subprocess, sys, tempfile, time, uuid
import numpy as np
BUILD = 'M3S4-2026-10-06-v9'
DEFAULTS = dict(sizes=[128,256,512,1024], repeats=5, fft_batch=500, seed=2026, input_seed=12345,
                files=200, bytes_per_file=4096, fsync=False, heat_points=128,
                heat_steps=100, checkpoint_interval=10, failure_step=57,
                noise_sigma=0.01, input_elements=50000, mpi_ranks=4, mpi_block=1024,
                weather_shape=[4,32,32], weather_slice=2, compression=None, blas_sizes=[128,256,512], blas_threads=1, lapack_points=128,
                rod_left=80.0, rod_right=20.0, rod_source=10.0)

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def save(path,data):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(data,indent=2)); tmp.replace(path)

def checked(argv,cwd,timeout=120,env=None):
    p=subprocess.run([str(x) for x in argv],cwd=cwd,text=True,capture_output=True,timeout=timeout,env=env)
    if p.returncode: raise RuntimeError(f'{argv}: rc={p.returncode}\n{p.stdout}\n{p.stderr}')
    return p.stdout

def validate(s):
    if not s['blas_sizes'] or any(type(n)is not int or not 16<=n<=1024 for n in s['blas_sizes']): raise ValueError('blas_sizes: integers 16..1024')
    if type(s['blas_threads']) is not int or not 1<=s['blas_threads']<=8: raise ValueError('blas_threads: integer 1..8')
    if type(s['lapack_points']) is not int or not 2<=s['lapack_points']<=512: raise ValueError('lapack_points: integer 2..512')
    if any(not math.isfinite(s[k]) for k in ['rod_left','rod_right','rod_source']): raise ValueError('rod parameters must be finite')
    for k in ['repeats','fft_batch','files','bytes_per_file','heat_points','heat_steps',
              'checkpoint_interval','input_elements','mpi_ranks','mpi_block']:
        if type(s[k]) is not int or s[k]<1: raise ValueError(f'{k} must be a positive integer')
    if not s['sizes'] or any(type(n)is not int or not 16<=n<=4096 for n in s['sizes']): raise ValueError('sizes: integers 16..4096')
    if s['heat_points']<3 or not 1<=s['failure_step']<s['heat_steps']: raise ValueError('invalid heat grid/failure step')
    if s['mpi_ranks']>64 or s['mpi_block']>1048576: raise ValueError('MPI classroom limits exceeded')
    if len(s['weather_shape'])!=3 or any(type(n)is not int or n<1 for n in s['weather_shape']): raise ValueError('weather_shape must have three positive dimensions')
    if not 0<=s['weather_slice']<s['weather_shape'][0]: raise ValueError('weather_slice outside time dimension')
    if s['compression'] not in [None,'gzip']: raise ValueError('compression must be None or gzip')
    if s['noise_sigma']<0 or not math.isfinite(s['noise_sigma']): raise ValueError('invalid noise_sigma')
    for k in ['seed','input_seed']:
        if type(s[k])is not int or s[k]<0: raise ValueError(k+' must be nonnegative integer')
    if s['files']>5000 or s['repeats']>20 or s['heat_steps']>10000 or s['fft_batch']>100000 or s['input_elements']>2_000_000: raise ValueError('classroom work limit exceeded')
    if s['files']*s['bytes_per_file']>64*1024*1024 or np.prod(s['weather_shape'])>2_000_000: raise ValueError('classroom data limit exceeded')

def summary(samples):
    return dict(samples_s=samples,median_s=statistics.median(samples),min_s=min(samples),max_s=max(samples))

def fft_lab(s,root):
    # Both direct DFT and FFTW execute compiled code through ctypes. Planning is separate.
    compiler=shutil.which('gcc')
    if not compiler: return dict(status='SKIPPED',reason='gcc unavailable')
    # find_library may return a SONAME from linker/compiler metadata even when
    # the running Python process cannot resolve it. Prefer module absolute paths.
    candidates=library_candidates('fftw3','EBROOTFFTW')
    for directory in os.getenv('LD_LIBRARY_PATH','').split(os.pathsep):
        if directory:
            candidates += [str(v) for v in sorted(Path(directory).glob('libfftw3.so*'))]
    candidates=list(dict.fromkeys(candidates))
    if not candidates: return dict(status='SKIPPED',reason='No FFTW3 shared library discovered')
    errors=[];fftw=None;libname=None
    for candidate in candidates:
        try:
            fftw=ct.CDLL(candidate);libname=candidate;break
        except OSError as e: errors.append(str(e))
    if fftw is None:
        raise RuntimeError('FFTW discovered but not loadable. EBROOTFFTW='+str(os.getenv('EBROOTFFTW'))+
                           '; candidates='+str(candidates)+'; errors='+str(errors))
    source=Path(__file__).with_name('m3s4_dft.c'); binary=root/'direct_dft.so'
    command=[compiler,'-O3','-Wall','-Wextra','-Werror','-fPIC','-shared',str(source),'-lm','-o',str(binary)]
    checked(command,root)
    direct=ct.CDLL(str(binary)); direct.direct_dft.argtypes=[ct.c_int,ct.c_void_p,ct.c_void_p]; direct.direct_dft.restype=None
    fftw.fftw_plan_dft_r2c_1d.argtypes=[ct.c_int,ct.c_void_p,ct.c_void_p,ct.c_uint]; fftw.fftw_plan_dft_r2c_1d.restype=ct.c_void_p
    fftw.fftw_execute.argtypes=[ct.c_void_p]; fftw.fftw_execute.restype=None
    fftw.fftw_destroy_plan.argtypes=[ct.c_void_p]; fftw.fftw_destroy_plan.restype=None
    rows=[]
    for n in s['sizes']:
        j=np.arange(n); x=np.ascontiguousarray(np.sin(2*np.pi*3*j/n)+0.5*np.sin(2*np.pi*7*j/n),dtype=np.float64)
        ref=np.empty(n,dtype=np.complex128); spectrum=np.empty(n//2+1,dtype=np.complex128)
        t=time.perf_counter(); plan=fftw.fftw_plan_dft_r2c_1d(n,x.ctypes.data,spectrum.ctypes.data,64) # FFTW_ESTIMATE
        planning=time.perf_counter()-t
        if not plan: raise RuntimeError('FFTW planning failed')
        try:
            def dft(): direct.direct_dft(n,x.ctypes.data,ref.ctypes.data)
            def fft():
                for _ in range(s['fft_batch']): fftw.fftw_execute(plan)
            dft(); fft() # warmup, excluded
            assert np.allclose(ref[:n//2+1],spectrum,rtol=1e-9,atol=1e-8)
            assert np.allclose(spectrum,np.fft.rfft(x),rtol=1e-9,atol=1e-8)
            ds=[]; fs=[]
            for r in range(s['repeats']):
                for label,fn in ([('d',dft),('f',fft)] if r%2==0 else [('f',fft),('d',dft)]):
                    t=time.perf_counter(); fn(); elapsed=time.perf_counter()-t
                    (ds if label=='d' else fs).append(elapsed if label=='d' else elapsed/s['fft_batch'])
            error=float(np.max(np.abs(ref[:n//2+1]-spectrum)))
            assert np.allclose(ref[:n//2+1],spectrum,rtol=1e-9,atol=1e-8)
            np.savez(root/f'spectrum_{n}.npz',signal=x,spectrum=spectrum)
            rows.append(dict(n=n,dft=summary(ds),fftw=summary(fs),planning_s=planning,max_error=error,
                             strongest_bins=np.argsort(np.abs(spectrum))[-2:][::-1].tolist()))
        finally: fftw.fftw_destroy_plan(plan)
    return dict(status='PASS',rows=rows,compiler=checked([compiler,'--version'],root).splitlines()[0],
                compiler_command=command,fftw_library=libname,fftw_version=ct.string_at(ct.addressof((ct.c_char*1).in_dll(fftw,'fftw_version'))).decode(),
                note='Serial. Forward unnormalized transforms; compare N/2+1 real-input bins. FFT timing includes amortized Python-call overhead, excludes planning.')

def io_lab(s,root):
    # Distinct records expose missing, reordered or corrupt content.
    payloads=[bytes([i%251])*s['bytes_per_file'] for i in range(s['files'])]
    expected=hashlib.sha256(b''.join(payloads)).hexdigest(); samples={'many':[],'one':[]}
    for r in range(s['repeats']):
        for case in (['many','one'] if r%2==0 else ['one','many']):
            with tempfile.TemporaryDirectory(prefix=f'io-{case}-',dir=root) as tmp:
                d=Path(tmp); t=time.perf_counter()
                if case=='many':
                    for i,payload in enumerate(payloads):
                        with open(d/f'{i:06d}.bin','wb') as f:
                            f.write(payload)
                            if s['fsync']: f.flush(); os.fsync(f.fileno())
                    samples[case].append(time.perf_counter()-t)
                    paths=sorted(d.glob('*.bin'))
                else:
                    with open(d/'all.bin','wb') as f:
                        for payload in payloads: f.write(payload)
                        if s['fsync']: f.flush(); os.fsync(f.fileno())
                    samples[case].append(time.perf_counter()-t)
                    paths=[d/'all.bin']
                assert len(paths)==(s['files'] if case=='many' else 1)
                assert sum(p.stat().st_size for p in paths)==s['files']*s['bytes_per_file']
                h=hashlib.sha256()
                for p in paths: h.update(p.read_bytes())
                assert h.hexdigest()==expected
    return dict(status='PASS',many=summary(samples['many']),one=summary(samples['one']),bytes=s['files']*s['bytes_per_file'],
                payload_sha256=expected,path=str(root),fsync=s['fsync'],
                note='Single writer, same ordered payload, creation/writes/close timed; directory creation, readback and removal excluded. Warm page cache; no physical disk bandwidth claim. fsync=False does not measure durable checkpoint cost.')

def heat_step(a):
    out=a.copy(); out[1:-1]=a[1:-1]+0.2*(a[:-2]-2*a[1:-1]+a[2:]); return out

def heat_child(config,root,mode):
    s=json.loads(Path(config).read_text()); root=Path(root); n=s['heat_points']
    identity={'points':n,'alpha':0.2,'steps':s['heat_steps']}; cp=root/'checkpoint.json'; start=0
    a=np.sin(np.linspace(0,np.pi,n)); a[0]=a[-1]=0
    if mode=='resume':
        state=json.loads(cp.read_text()); assert state['schema']==1 and state['parameters']==identity
        a=np.array(state['grid'],dtype=float); start=state['step']
        assert a.shape==(n,) and np.all(np.isfinite(a)) and 0<=start<=s['heat_steps']
        assert hashlib.sha256(a.tobytes()).hexdigest()==state['grid_sha256']
    writes=0; checkpoint_s=0.0
    for step in range(start+1,s['heat_steps']+1):
        a=heat_step(a)
        if mode=='fail' and step==s['failure_step']:
            save(root/'failure.json',{'failure_step':step,'last_checkpoint':(step-1)//s['checkpoint_interval']*s['checkpoint_interval'],'checkpoint_writes':writes,'checkpoint_seconds':checkpoint_s})
            os._exit(17) # controlled process termination; latest complete checkpoint survives
        if mode=='fail' and step%s['checkpoint_interval']==0:
            t=time.perf_counter(); save(cp,{'schema':1,'step':step,'parameters':identity,'grid':a.tolist(),'grid_sha256':hashlib.sha256(a.tobytes()).hexdigest()})
            checkpoint_s+=time.perf_counter()-t; writes+=1
    np.save(root/(mode+'_final.npy'),a)
    save(root/(mode+'_stats.json'),{'start_step':start,'checkpoint_writes':writes,'checkpoint_seconds':checkpoint_s})

def checkpoint_lab(s,root,config):
    cmd=[sys.executable,str(Path(__file__).resolve()),'--heat-child',str(config),str(root)]
    checked(cmd+['reference'],root)
    ref=np.load(root/'reference_final.npy').copy()
    p=subprocess.run(cmd+['fail'],cwd=root,capture_output=True,text=True,timeout=120)
    assert p.returncode==17, p.stderr
    cp=root/'checkpoint.json'; loss=json.loads((root/'failure.json').read_text())
    if cp.exists():
        checked(cmd+['resume'],root); restarted=root/'resume_final.npy'; resumed=json.loads((root/'resume_stats.json').read_text())['start_step']
    else:
        # Failure before first checkpoint: restart from initial state, explicitly recorded.
        checked(cmd+['reference'],root); restarted=root/'reference_final.npy'; resumed=0
    actual=np.load(restarted)
    assert np.array_equal(ref,actual)
    return dict(status='PASS',failure_exit_code=17,failure_step=s['failure_step'],resumed_from_step=resumed,
                completed_steps_lost=s['failure_step']-1-resumed,final_identical=True,
                checkpoint_bytes=cp.stat().st_size if cp.exists() else 0,checkpoint_writes=loss['checkpoint_writes'],checkpoint_seconds=loss['checkpoint_seconds'],
                note='Application checkpoint: step, parameters, full grid and integrity hash. Separate processes demonstrate restart. Atomic rename protects incomplete files; this classroom version does not fsync or promise power-loss durability.')

def linalg_reference(root):
    compiler=shutil.which('gcc')
    if not compiler: return None, None
    cmd=[compiler,'-O3','-march=native','-Wall','-Wextra','-Werror','-shared','-fPIC',
         str(Path(__file__).with_name('m3s4_linalg.c')),'-o',str(root/'manual_linalg.so')]
    checked(cmd,root); lib=ct.CDLL(str(root/'manual_linalg.so'))
    for name in ['matmul_ijk','matmul_ikj']:
        f=getattr(lib,name); f.argtypes=[ct.c_int,ct.c_void_p,ct.c_void_p,ct.c_void_p]; f.restype=None
    lib.gaussian.argtypes=[ct.c_int,ct.c_void_p,ct.c_void_p,ct.c_int];lib.gaussian.restype=ct.c_int
    return lib,cmd

def library_candidates(package,envroot):
    paths=[]
    if os.getenv(envroot):
        root=Path(os.environ[envroot])
        for folder in ['lib','lib64']:
            paths.extend(str(p) for p in sorted((root/folder).glob('lib'+package+'.so*')))
    found=ctypes.util.find_library(package)
    if found: paths.append(found)
    return list(dict.fromkeys(paths))

def load_linalg(symbol,lapack=False):
    errors=[]
    packages=[('openblas','EBROOTOPENBLAS')]
    if lapack: packages.append(('lapack','EBROOTLAPACK'))
    else: packages.append(('blas','EBROOTBLAS'))
    for package,env in packages:
        for path in library_candidates(package,env):
            try:
                lib=ct.CDLL(path)
                if hasattr(lib,'openblas_get_config'):
                    lib.openblas_get_config.restype=ct.c_char_p
                    config=lib.openblas_get_config().decode()
                    if 'USE64BITINT' in config:
                        errors.append(path+': ILP64 not supported by this LP64 example');continue
                else: config='System LP64 '+package+'; implementation/version not reported'
                getattr(lib,symbol)
                return lib,path,config
            except (OSError,AttributeError) as e: errors.append(path+': '+str(e))
    return None,None,'; '.join(errors) or 'No suitable shared library found'

def blas_lab(s,root):
    manual,cmd=linalg_reference(root)
    if manual is None: return dict(status='SKIPPED',reason='gcc unavailable')
    lib,path,config=load_linalg('cblas_dgemm')
    if lib is None: return dict(status='SKIPPED',reason=config)
    f=lib.cblas_dgemm
    f.argtypes=[ct.c_int]*6+[ct.c_double,ct.c_void_p,ct.c_int,ct.c_void_p,ct.c_int,ct.c_double,ct.c_void_p,ct.c_int]
    f.restype=None
    requested=s['blas_threads']; setter=getattr(lib,'openblas_set_num_threads',None)
    getter=getattr(lib,'openblas_get_num_threads',None)
    if setter: setter.argtypes=[ct.c_int];setter.restype=None
    if getter: getter.restype=ct.c_int
    if requested>1 and (setter is None or getter is None):
        return dict(status='SKIPPED',reason='Thread experiment requires OpenBLAS thread controls; use blas_threads=1 for system BLAS')
    available=int(os.getenv('SLURM_CPUS_PER_TASK',str(len(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else 1)))
    if requested>available: raise ValueError('blas_threads exceeds allocated/available CPUs')
    rows=[];rng=np.random.default_rng(s['seed'])
    try:
        for n in s['blas_sizes']:
            # Same seeded dense inputs, both manual baselines in compiled C.
            a=rng.normal(size=(n,n));b=rng.normal(size=(n,n));c=np.empty_like(a)
            def product(): f(101,111,111,n,n,n,1.0,a.ctypes.data,n,b.ctypes.data,n,0.0,c.ctypes.data,n)
            if setter: setter(1)
            if getter and getter()!=1: raise RuntimeError('Could not enforce single-thread BLAS')
            product(); reference=c.copy();samples={'manual_ijk':[],'manual_ikj':[],'blas_1':[]};errors={}
            funcs={'manual_ijk':lambda:manual.matmul_ijk(n,a.ctypes.data,b.ctypes.data,c.ctypes.data),
                   'manual_ikj':lambda:manual.matmul_ikj(n,a.ctypes.data,b.ctypes.data,c.ctypes.data),'blas_1':product}
            for name,fn in funcs.items():
                fn();assert np.allclose(c,reference,rtol=1e-10,atol=1e-10),name+' result mismatch'
                errors[name]=float(np.max(np.abs(c-reference)))
            # Rotate order; validation/allocations/random numbers are outside timing.
            names=list(funcs)
            for r in range(s['repeats']):
                for name in names[r%3:]+names[:r%3]:
                    start=time.perf_counter();funcs[name]();samples[name].append(time.perf_counter()-start)
                    assert np.allclose(c,reference,rtol=1e-10,atol=1e-10)
            actual=1 if getter else None
            if requested>1:
                setter(requested);actual=getter()
                if actual!=requested: raise RuntimeError(f'OpenBLAS configured {actual}, requested {requested}')
                product();samples['blas_requested']=[]
                for _ in range(s['repeats']):
                    start=time.perf_counter();product();samples['blas_requested'].append(time.perf_counter()-start)
                    assert np.allclose(c,reference,rtol=1e-10,atol=1e-10)
            rows.append(dict(n=n,timings={k:summary(v) for k,v in samples.items()},max_abs_errors=errors,
                             requested_threads=requested,configured_threads=actual,
                             blas_speedup_over_ijk=statistics.median(samples['manual_ijk'])/statistics.median(samples['blas_1']),
                             blas_speedup_over_ikj=statistics.median(samples['manual_ikj'])/statistics.median(samples['blas_1'])))
    finally:
        if setter: setter(1)
    return dict(status='PASS',library=path,configuration=config,compile_command=cmd,rows=rows,
                note='Dense C=A B can represent applying many sensor/calibration transforms. All methods compiled; warmup and repeated timings. CBLAS row-major, no transpose, alpha=1 beta=0. Thread getter reports configured threads, not measured active workers. System BLAS may be unoptimized; speedup is measured, never guaranteed.')

def lapack_lab(s,root):
    manual,cmd=linalg_reference(root)
    if manual is None: return dict(status='SKIPPED',reason='gcc unavailable')
    lib,path,config=load_linalg('dgesv_',lapack=True)
    if lib is None: return dict(status='SKIPPED',reason=config)
    setter=getattr(lib,'openblas_set_num_threads',None)
    if setter: setter(1)
    solve=lib.dgesv_;solve.argtypes=[ct.c_void_p]*8;solve.restype=None
    def lapack(a,b):
        # Fortran expects column-major storage; DGESV overwrites both buffers.
        aa=np.array(a,order='F',copy=True);bb=b.copy();n=ct.c_int(len(b));one=ct.c_int(1)
        piv=np.empty(len(b),dtype=np.int32);info=ct.c_int()
        start=time.perf_counter()
        solve(ct.byref(n),ct.byref(one),aa.ctypes.data,ct.byref(n),piv.ctypes.data,bb.ctypes.data,ct.byref(n),ct.byref(info))
        return bb,info.value,time.perf_counter()-start,piv.tolist()
    def teaching(a,b,pivot=True):
        aa=np.array(a,order='C',copy=True);bb=b.copy();start=time.perf_counter()
        info=manual.gaussian(len(b),aa.ctypes.data,bb.ctypes.data,int(pivot))
        return bb,info,time.perf_counter()-start
    n=s['lapack_points'];left=s['rod_left'];right=s['rod_right'];source=s['rod_source']
    dx=1/(n+1);a=2*np.eye(n)-np.eye(n,k=1)-np.eye(n,k=-1)
    b=np.full(n,source*dx*dx);b[0]+=left;b[-1]+=right
    xx=np.arange(1,n+1)*dx;analytic=left+(right-left)*xx+0.5*source*xx*(1-xx)
    samples={'manual_pivot':[],'lapack':[]};results={}
    for method,fn in [('manual_pivot',teaching),('lapack',lapack)]:
        x,info,*_=fn(a,b);assert info==0
        for _ in range(s['repeats']):
            x,info,seconds,*_=fn(a,b);assert info==0;samples[method].append(seconds)
        residual=float(np.linalg.norm(a@x-b,np.inf)/(np.linalg.norm(a,np.inf)*np.linalg.norm(x,np.inf)+np.linalg.norm(b,np.inf)))
        assert residual<1e-12 and np.allclose(x,analytic,rtol=1e-9,atol=1e-9)
        results[method]=dict(relative_residual=residual,max_analytic_error=float(np.max(np.abs(x-analytic))))
    # Nonsingular matrix with zero leading diagonal: row exchange is necessary.
    tricky=np.array([[0.,1.],[1.,1.]]);rhs=np.array([1.,2.])
    _,nopivot,_=teaching(tricky,rhs,False);xp,pinfo,_=teaching(tricky,rhs);xl,linfo,_,piv=lapack(tricky,rhs)
    assert nopivot>0 and pinfo==linfo==0 and np.allclose(xp,[1,1]) and np.allclose(xl,[1,1])
    _,singular,_,_=lapack(np.array([[1.,2.],[2.,4.]]),np.array([3.,6.]))
    assert singular>0
    return dict(status='PASS',library=path,configuration=config,compile_command=cmd,points=n,
                boundaries=[left,right],source=source,timings={k:summary(v) for k,v in samples.items()},checks=results,
                temperatures=x.tolist(),positions=xx.tolist(),pivot_example=dict(no_pivot_info=nopivot,lapack_info=linfo,pivots=piv,solution=xl.tolist()),
                singular_info=singular,note='Steady rod: -T double-prime = source, unit length, fixed end temperatures. Both dense solvers use partial pivoting. DGESV uses 32-bit LAPACK integers, column-major A; INFO=0 success, >0 singular, <0 invalid argument. Copies outside timing. A tridiagonal solver would exploit this rod structure better than dense DGESV; dense solve is for learning the library interface.')


def hdf5_runtime_environment():
    # Compiler wrappers can find HDF5 while the dynamic loader misses its
    # indirect Szip/libaec dependencies. A RUNPATH on the executable alone
    # does not resolve every dependency of libhdf5. Use only directories
    # supplied by the current loaded module/toolchain environment.
    env=os.environ.copy();directories=[]
    for variable in ['LD_LIBRARY_PATH','LIBRARY_PATH']:
        directories += [v for v in env.get(variable,'').split(os.pathsep) if v]
    # Prefer the selected HDF5 and compression roots before other loaded roots.
    roots=['EBROOTHDF5','EBROOTSZIP','EBROOTLIBAEC']
    roots += sorted(k for k in env if k.startswith('EBROOT') and k not in roots)
    for key in roots:
        if env.get(key):
            for folder in ['lib','lib64']:
                candidate=Path(env[key])/folder
                if candidate.is_dir(): directories.append(str(candidate))
    directories=list(dict.fromkeys(v for v in directories if Path(v).is_dir()))
    if directories: env['LD_LIBRARY_PATH']=os.pathsep.join(directories)
    return env,directories

def hdf5_lab(s,root):
    try: import h5py
    except ImportError:
        # Parallel builds commonly provide h5pcc instead of h5cc.
        runtime_env,runtime_dirs=hdf5_runtime_environment()
        hdfroot=Path(os.environ['EBROOTHDF5']) if os.getenv('EBROOTHDF5') else None
        wrappers=[shutil.which('h5cc'),shutil.which('h5pcc')]
        if hdfroot: wrappers += [str(hdfroot/'bin'/name) for name in ['h5cc','h5pcc'] if (hdfroot/'bin'/name).is_file()]
        source=str(Path(__file__).with_name('m3s4_hdf5.c'));exe=str(root/'hdf5_demo')
        attempts=[[w,source,'-o',exe] for w in dict.fromkeys(wrappers) if w]
        if hdfroot:
            cc=shutil.which('mpicc') or shutil.which('gcc')
            for folder in ['lib','lib64']:
                libdir=hdfroot/folder
                if cc and (libdir/'libhdf5.so').exists():
                    attempts.append([cc,source,'-I'+str(hdfroot/'include'),'-L'+str(libdir),
                                     '-Wl,-rpath,'+str(libdir),'-lhdf5','-o',exe])
        if not attempts: return dict(status='SKIPPED',reason='No h5py, h5cc/h5pcc or usable EBROOTHDF5 compiler path')
        errors=[]
        for cmd in attempts:
            # Some wrapper installations record dependency -L paths without
            # exporting every dependency as an EasyBuild module variable.
            if Path(cmd[0]).name in ['h5cc','h5pcc']:
                show=subprocess.run([cmd[0],'-show'],cwd=root,capture_output=True,text=True,env=runtime_env,timeout=30)
                if show.returncode==0:
                    flags=shlex.split(show.stdout);extra=[]
                    for i,flag in enumerate(flags):
                        if flag=='-L' and i+1<len(flags): extra.append(flags[i+1])
                        elif flag.startswith('-L') and len(flag)>2: extra.append(flag[2:])
                    runtime_dirs=list(dict.fromkeys(runtime_dirs+[d for d in extra if Path(d).is_dir()]))
                    if runtime_dirs: runtime_env['LD_LIBRARY_PATH']=os.pathsep.join(runtime_dirs)
            try: checked(cmd,root,env=runtime_env);break
            except RuntimeError as e: errors.append(str(e))
        else: raise RuntimeError('HDF5 found but compilation failed: '+ '\n'.join(errors))
        args=[exe,*map(str,s['weather_shape']),str(s['weather_slice']),'1' if s['compression']=='gzip' else '0']
        loader=None
        if shutil.which('ldd'):
            loader=checked([shutil.which('ldd'),exe],root,env=runtime_env)
            if 'not found' in loader:
                raise RuntimeError('HDF5 runtime dependencies missing after module path repair:\n'+loader+
                                   '\nLoad the matching Szip/libaec module for the HDF5 toolchain; runtime dirs: '+str(runtime_dirs))
        result=json.loads(checked(args,root,env=runtime_env));assert result['passed']
        assert result['shape']==s['weather_shape'] and result['slice_time']==s['weather_slice']
        return dict(status='PASS',backend='C HDF5 serial writer',result=result,command=cmd,run_command=args,
                    bytes=(root/'weather.h5').stat().st_size,runtime_library_dirs=runtime_dirs,loader_dependencies=loader,
                    note='Editable shape, one-plane chunks, units, time axis, compression and hyperslab. Full values and time-axis readback verified. h5pcc selects a parallel-capable library; this program still uses one writer.')
    shape=tuple(s['weather_shape']); t,y,x=np.indices(shape); data=273.15+t+0.1*y+0.01*x
    path=root/'weather.h5'; start=time.perf_counter()
    with h5py.File(path,'w') as f:
        ds=f.create_dataset('temperature',data=data,chunks=(1,shape[1],shape[2]),compression=s['compression'])
        ds.attrs['units']='K'; ds.attrs['dimensions']='time,y,x'
        f.create_dataset('time',data=np.arange(shape[0])); f['time'].attrs['units']='hours since simulation start'
    write_s=time.perf_counter()-start
    with h5py.File(path,'r') as f:
        ds=f['temperature']; assert ds.shape==shape and ds.attrs['units']=='K'
        assert np.array_equal(ds[:],data)
        start=time.perf_counter(); region=ds[s['weather_slice'],:,:]; read_s=time.perf_counter()-start
        assert np.array_equal(region,data[s['weather_slice']])
        info=dict(shape=list(ds.shape),chunks=list(ds.chunks),compression=ds.compression,units=ds.attrs['units'])
    return dict(status='PASS',backend='h5py serial HDF5',h5py=h5py.__version__,hdf5=h5py.version.hdf5_version,
                dataset=info,bytes=path.stat().st_size,write_close_s=write_s,read_slice_s=read_s,
                note='Full readback plus selected time slice. Serial HDF5, not parallel HDF5; NetCDF would add standardized dimensions/conventions. No compression speedup assumed.')

def mpi_lab(s,root):
    cc=shutil.which('mpicc')
    if not cc: return dict(status='SKIPPED',reason='mpicc unavailable; load MPI toolchain')
    inside=bool(os.getenv('SLURM_JOB_ID'))
    allocated=int(os.getenv('SLURM_NTASKS','1') or '1')
    if inside and allocated<s['mpi_ranks']:
        return dict(status='SKIPPED',reason=f'current allocation has {allocated} task(s); MPI-IO needs {s["mpi_ranks"]}. Submit the notebook batch job instead of reusing the Jupyter allocation.')
    source=Path(__file__).with_name('m3s4_mpiio.c'); binary=root/'mpiio'
    compile_cmd=[cc,'-O2','-Wall','-Wextra','-Werror',str(source),'-o',str(binary)]; checked(compile_cmd,root)

    launchers=[]
    if inside and shutil.which('srun'):
        launchers.append([shutil.which('srun'),'--exact','--ntasks='+str(s['mpi_ranks'])])
    if shutil.which('mpiexec'):
        launchers.append([shutil.which('mpiexec'),'-n',str(s['mpi_ranks'])])
    if not launchers: return dict(status='SKIPPED',reason='no usable MPI launcher (srun/mpiexec)')

    attempts=[]; text=None; command=None
    for prefix in launchers:
        (root/'mpi_output.bin').unlink(missing_ok=True)
        command=prefix+[str(binary),str(s['mpi_block'])]
        p=subprocess.run([str(x) for x in command],cwd=root,text=True,capture_output=True,timeout=120)
        attempts.append(dict(command=command,returncode=p.returncode,stdout=p.stdout,stderr=p.stderr))
        if p.returncode==0:
            text=p.stdout
            break
    if text is None:
        detail='\\n\\n'.join(
            'command: '+repr(a['command'])+'\\nrc='+str(a['returncode'])+'\\nstdout:\\n'+a['stdout']+'\\nstderr:\\n'+a['stderr']
            for a in attempts
        )
        raise RuntimeError('MPI launch failed with all available launchers:\\n'+detail)

    result=json.loads(next(line for line in text.splitlines() if line.startswith('{')))
    assert result['passed'] and result['ranks']==s['mpi_ranks']
    assert result['bytes']==s['mpi_ranks']*s['mpi_block']
    contents=(root/'mpi_output.bin').read_bytes()
    expected=b''.join(bytes((r*17+j)%251 for j in range(s['mpi_block'])) for r in range(s['mpi_ranks']))
    assert contents==expected
    return dict(status='PASS',command=command,attempts=attempts,compiler_command=compile_cmd,result=result,
                note='Real MPI processes, collective write_at_all to nonoverlapping byte ranges, sync/close, reopen and collective readback. Demonstrates correctness, not proof of scalability on a parallel filesystem.')

def simulation(data,seed,sigma):
    noise=np.random.default_rng(seed).normal(0,sigma,size=data.shape)
    return float(np.mean((data+noise)**2))

def replay(manifest_path):
    path=Path(manifest_path).resolve(); m=json.loads(path.read_text()); root=path.parent
    assert sha(Path(__file__))==m['sources']['m3s4_lab.py'], 'runner source changed'
    inp=root/m['input']['path']; assert sha(inp)==m['input']['sha256'], 'input checksum mismatch'
    params=m['parameters']; result=simulation(np.load(inp),params['seed'],params['noise_sigma'])
    assert math.isclose(result,m['result']['value'],rel_tol=1e-12,abs_tol=1e-12), 'scientific result differs'
    return dict(status='PASS',recorded=m['result']['value'],replayed=result,
                note='Numerical agreement within tolerance; replay timings are not expected to match.')

def main(config,root):
    root=Path(root).resolve(); root.mkdir(parents=True,exist_ok=True)
    if (root/'report.json').exists(): raise RuntimeError('Use a new run directory; report already exists')
    s=json.loads(Path(config).read_text()); validate(s); save(root/'settings.json',s); config=root/'settings.json'
    report=dict(build=BUILD,settings=s,run_id=root.name,experiments={})
    for name,fn in [('blas',lambda:blas_lab(s,root)),('lapack',lambda:lapack_lab(s,root)),('fft',lambda:fft_lab(s,root)),('io',lambda:io_lab(s,root)),
                    ('checkpoint',lambda:checkpoint_lab(s,root,config)),('hdf5',lambda:hdf5_lab(s,root)),('mpiio',lambda:mpi_lab(s,root))]:
        try: report['experiments'][name]=fn()
        except Exception as e: report['experiments'][name]=dict(status='FAIL',reason=f'{type(e).__name__}: {e}')
        print(name,report['experiments'][name]['status'],flush=True); save(root/'report.json',report)
    data=np.random.default_rng(s['input_seed']).normal(size=s['input_elements']); np.save(root/'input.npy',data)
    samples=[]; result=None
    for _ in range(s['repeats']):
        t=time.perf_counter(); result=simulation(data,s['seed'],s['noise_sigma']); samples.append(time.perf_counter()-t)
    bad=[simulation(data,int.from_bytes(os.urandom(8),'little'),s['noise_sigma']) for _ in range(2)]
    sources={p.name:sha(p) for p in Path(__file__).parent.glob('m3s4_*') if p.suffix in ['.py','.c','.sbatch']}
    git=subprocess.run(['git','rev-parse','HEAD'],cwd=Path(__file__).parent,capture_output=True,text=True)
    df=subprocess.run(['df','-T',str(root)],capture_output=True,text=True) if shutil.which('df') else None
    manifest=dict(build=BUILD,run_id=root.name,sources=sources,code_commit=git.stdout.strip() if git.returncode==0 else None,
                  input=dict(path='input.npy',sha256=sha(root/'input.npy'),elements=s['input_elements']),parameters=s,
                  command=[sys.executable,str(Path(__file__).resolve()),'--config','settings.json','--out',str(root)],
                  replay_command=[sys.executable,str(Path(__file__).resolve()),'--replay','manifest.json'],
                  software=dict(python=sys.version,numpy=np.__version__,modules=[m for m in os.getenv('LOADEDMODULES','').split(':') if m],
                                blas=report['experiments']['blas'],lapack=report['experiments']['lapack'],
                                fftw=report['experiments']['fft'],hdf5=report['experiments']['hdf5']),
                  system=dict(host=platform.node(),platform=platform.platform(),cpus=sorted(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else None,
                              filesystem=df.stdout if df else None),
                  resources={k:os.getenv(k) for k in ['SLURM_JOB_ID','SLURM_JOB_NUM_NODES','SLURM_NTASKS','SLURM_CPUS_PER_TASK','SLURM_JOB_PARTITION','SLURM_JOB_NODELIST','OMP_NUM_THREADS']},
                  result=dict(value=result,measurement=summary(samples)),
                  outputs={p.name:sha(p) for p in root.iterdir() if p.is_file() and p.name not in ['report.json','manifest.json']})
    save(root/'manifest.json',manifest); report['experiments']['replay']=replay(root/'manifest.json')
    report['unseeded_results']=bad
    report['experiments']['replay']['changed_seed_result']=simulation(data,s['seed']+1,s['noise_sigma'])
    report['experiments']['replay']['changed_sigma_result']=simulation(data,s['seed'],s['noise_sigma']*2)
    report['failed']=[k for k,v in report['experiments'].items() if v['status']=='FAIL']
    report['skipped']=[k for k,v in report['experiments'].items() if v['status']=='SKIPPED']
    report['core_pass']=all(report['experiments'][k]['status']=='PASS' for k in ['io','checkpoint','replay']) and not report['failed']
    save(root/'report.json',report); return 0 if report['core_pass'] else 1

if __name__=='__main__':
    if '--heat-child' in sys.argv: heat_child(*sys.argv[2:]); sys.exit(0)
    parser=argparse.ArgumentParser(); parser.add_argument('--config'); parser.add_argument('--out'); parser.add_argument('--replay')
    args=parser.parse_args()
    if args.replay: print(json.dumps(replay(args.replay),indent=2))
    elif args.config and args.out: sys.exit(main(args.config,args.out))
    else: parser.error('--config and --out required, or --replay')


'''
SOURCES['m3s4_lab.sbatch'] = r'''#!/bin/bash
#SBATCH --job-name=m3s4_lab
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=4
#SBATCH --cpus-per-task=1
#SBATCH --mem=4G
#SBATCH --time=00:10:00
set -eo pipefail
# Arguments: runner settings run-directory Python executable (captured from kernel).
runner=$1
settings=$2
run_dir=$3
kernel_python=$4
if ! command -v module >/dev/null 2>&1 && [ -r /cvmfs/software.eessi.io/versions/2023.06/init/lmod/bash ]; then
  source /cvmfs/software.eessi.io/versions/2023.06/init/lmod/bash
fi
if command -v module >/dev/null 2>&1; then
  module load foss/2023b
  # Optional packages: select a build compatible with the GCC/gompi 2023b toolchain.
  for package in FFTW HDF5; do
    candidate=$(module -t avail "$package" 2>&1 | sed 's/\x1b\[[0-9;]*m//g' | awk '/^(FFTW|HDF5)\// && /(13\.2\.0|2023b)/ {print $1; exit}')
    if [ -n "$candidate" ]; then module load "$candidate"; fi
  done
  module -t list > "$run_dir/modules.txt" 2>&1
fi
export OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1
# Python executes serial sections once; the runner launches an srun MPI step.
"$kernel_python" "$runner" --config "$settings" --out "$run_dir"
'''

SOURCE_DIR = WORKSPACE / 'sources'
SOURCE_DIR.mkdir(exist_ok=True)
for name, content in SOURCES.items():
    (SOURCE_DIR / name).write_text(content)
import importlib.util
spec = importlib.util.spec_from_file_location('m3s4_runner', SOURCE_DIR / 'm3s4_lab.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)
lab.validate(SETTINGS)
print('Sources ready:', ', '.join(sorted(SOURCES)))


### Submit or reuse a run
The notebook submits a fresh Slurm allocation when `sbatch` is available, even if Jupyter itself is already running inside a small Slurm allocation. The batch script initializes EESSI if available, loads `foss/2023b`, and tries compatible FFTW/HDF5 builds. It records modules **inside the job**. Edit the embedded batch source if your cluster uses a different toolchain. The original notebook Python executable is retained; loading a module in a child shell does not install packages into that interpreter.

`h5py` provides the editable weather dimensions/compression experiment. If only `h5cc` is available, the runner executes a checked C HDF5 example with fixed dimensions and clearly labels that limitation. Compilation or runtime failure is **FAIL**, not a silent skip.


In [ ]:
fingerprint = hashlib.sha256(json.dumps({'settings': SETTINGS, 'sources': SOURCES,
                                          'partition': PARTITION}, sort_keys=True).encode()).hexdigest()
state_path = WORKSPACE / ('state-' + fingerprint[:20] + '.json')
RUN_STATE = None
if state_path.exists() and not NEW_RUN:
    RUN_STATE = json.loads(state_path.read_text())
    print('Reusing run:', RUN_STATE['run_dir'], 'job:', RUN_STATE.get('job_id'))
else:
    run_dir = WORKSPACE.resolve() / ('run-' + uuid.uuid4().hex[:12])
    run_dir.mkdir()
    # Freeze sources in this run: later edits cannot alter a queued job.
    for name, content in SOURCES.items(): (run_dir / name).write_text(content)
    config = run_dir / 'settings.json'
    config.write_text(json.dumps(SETTINGS, indent=2))
    runner = run_dir / 'm3s4_lab.py'
    RUN_STATE = {'fingerprint': fingerprint, 'run_dir': str(run_dir), 'job_id': None}
    if RUN_LOCAL:
        p = subprocess.run([sys.executable, str(runner), '--config', str(config), '--out', str(run_dir)],
                           capture_output=True, text=True, timeout=600)
        print(p.stdout)
        if p.returncode:
            print(p.stderr)
            print('Local run completed with one or more failed experiments; inspect report.json in', run_dir)
    elif shutil.which('sbatch'):
        cmd = ['sbatch', '--parsable', '--partition=' + PARTITION, '--ntasks=' + str(SETTINGS['mpi_ranks']),
               '--cpus-per-task=' + str(SETTINGS['blas_threads']),
               '--output=' + str(run_dir / 'slurm-%j.out'), '--chdir=' + str(run_dir),
               str(run_dir / 'm3s4_lab.sbatch'), str(runner), str(config), str(run_dir), sys.executable]
        env = os.environ.copy()
        for key in ('SLURM_MEM_PER_CPU','SLURM_MEM_PER_GPU','SLURM_MEM_PER_NODE'):
            env.pop(key, None)
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=30, env=env)
        if p.returncode: raise RuntimeError(p.stderr)
        job_id = p.stdout.strip().split(';')[0]
        if not job_id.isdigit(): raise RuntimeError('Unexpected sbatch response: ' + p.stdout)
        RUN_STATE['job_id'] = job_id
        print('Submitted fresh Slurm job', job_id, 'with', SETTINGS['mpi_ranks'], 'MPI task slots')
    else:
        RUN_STATE = None
        print('NOT RUN: no Slurm. For a laptop, set RUN_LOCAL=True and rerun setup onwards.')
    if RUN_STATE:
        state_path.write_text(json.dumps(RUN_STATE, indent=2))


### Collect the completed report
If the job is still queued after the short wait, **rerun this collection cell**, not the submission cell. Later sections use only this run's verified report. Intermediate reports are not treated as completed jobs. A failed job must be investigated from its log; no replacement result is invented.

Confirm the printed **notebook build and report build agree**. Repeating collection for an old run never reruns its experiments. After downloading an updated notebook, restart the kernel and run setup → materialize → submit → collect.


In [ ]:
REPORT = None
if RUN_STATE:
    current_fingerprint = hashlib.sha256(json.dumps({'settings': SETTINGS, 'sources': SOURCES,
                                                    'partition': PARTITION}, sort_keys=True).encode()).hexdigest()
    if RUN_STATE.get('fingerprint') != current_fingerprint:
        raise RuntimeError('This RUN_STATE is from different sources/settings. Rerun setup, materialize and submit before collecting.')
    RUN_DIR = Path(RUN_STATE['run_dir'])
    deadline = time.monotonic() + WAIT_SECONDS
    while True:
        report_path = RUN_DIR / 'report.json'
        candidate = json.loads(report_path.read_text()) if report_path.exists() else {}
        if 'core_pass' in candidate:
            REPORT = candidate
            break
        if RUN_STATE.get('job_id') and shutil.which('squeue'):
            q = subprocess.run(['squeue', '-h', '-j', RUN_STATE['job_id']], capture_output=True, text=True, timeout=15)
            if q.returncode and 'Invalid job id specified' not in q.stderr:
                raise RuntimeError(q.stderr)
            if q.returncode or not q.stdout.strip():
                logs = list(RUN_DIR.glob('slurm-*.out'))
                print(logs[-1].read_text()[-6000:] if logs else 'No Slurm log found yet')
                raise RuntimeError('Job left queue without a completed report. Check sacct/log; rerun collection if files are delayed.')
        if time.monotonic() >= deadline:
            raise TimeoutError('Job still pending. Rerun ONLY this collection cell later.')
        time.sleep(2)
    print('Notebook build:', BUILD, '| Report build:', REPORT.get('build'), '| Run:', REPORT.get('run_id'))
    if REPORT.get('build') != BUILD:
        raise RuntimeError('Old report detected. Open the latest notebook, restart kernel, then run setup, materialize, submit and collect.')
    assert REPORT['settings'] == SETTINGS, 'Settings changed: materialize and submit a new run first'
    print('Run:', REPORT['run_id'])
    for name, result in REPORT['experiments'].items():
        print(name, result['status'], result.get('reason', ''))
    if not REPORT['core_pass']: raise RuntimeError('An experiment failed. Read report.json and job log.')
    manifest = json.loads((RUN_DIR / 'manifest.json').read_text())
    print('Evidence:', RUN_DIR / 'manifest.json')
else:
    print('NOT RUN. No benchmark results are available.')

def show_experiment(name):
    if REPORT is None:
        print('NOT RUN:', name)
        return None
    result = REPORT['experiments'][name]
    print(name, result['status'], result.get('reason', ''))
    return result if result['status'] == 'PASS' else None

# Presentation helpers: these display measured evidence; they do not rerun a job.
def lesson_text(text):
    from IPython.display import display, Markdown
    display(Markdown(text))

def lesson_table(headers, rows):
    def clean(value): return str(value).replace('|', '\\|').replace('\n',' ')
    lesson_text('| '+' | '.join(map(clean,headers))+' |\n| '+ ' | '.join(['---']*len(headers))+' |\n'+
                '\n'.join('| '+' | '.join(map(clean,row))+' |' for row in rows))

def lesson_details(title, value):
    from IPython.display import display, HTML
    import html
    text=value if isinstance(value,str) else json.dumps(value,indent=2)
    display(HTML('<details><summary>'+html.escape(title)+'</summary><pre>'+html.escape(text)+'</pre></details>'))

def lesson_code(*names):
    import ast
    source=SOURCES['m3s4_lab.py']; tree=ast.parse(source)
    snippets=[ast.get_source_segment(source,node) for node in tree.body
              if isinstance(node,(ast.FunctionDef,ast.AsyncFunctionDef)) and node.name in names]
    lesson_details('Inspect the actual code executed in this activity','\n\n'.join(snippets))

def lesson_plot(draw):
    try: import matplotlib.pyplot as plt
    except ImportError:
        print('Optional plot unavailable; the measured table and explanation remain usable.')
        return
    draw(plt)
    plt.show()


## 1 — Find the two hidden frequencies in a sensor signal
**Situation:** a sensor records a signal that combines 3 cycles and 7 cycles across our observation window. We want to recover those frequencies.

**What actually runs:** the worker creates the same signal for two compiled implementations: a manual C DFT and real FFTW. FFTW plans once, then executes repeatedly. Both outputs are checked against each other and NumPy. Python coordinates; it is not the slow numerical baseline.

**Look for:** two peaks at bins 3 and 7, matching results, and shorter FFTW execution time. The plot separates correctness (same spectrum) from performance (different cost). The table uses milliseconds for both methods.

**Try:** change one entry in `sizes`, e.g. 512 → 1024. Predict the change in time; rerun the job. A direct DFT needs roughly N² work, while an FFT needs roughly N log N. Actual timings also reflect caches and overhead. Planning is excluded from the speedup.

**Explain:** how can two algorithms produce the same answer using very different amounts of work? This FFTW experiment is serial; its speedup does not come from more threads.

### The operations executed
**Manual C:** each frequency examines every input sample (nested k/j loops). Core update:

```c
double angle = -tau * (double)k * j / n;
re += in[j] * cos(angle); im += in[j] * sin(angle);
```
**FFTW:** make a plan, then execute it; setup/type declarations are omitted here.

```python
t=time.perf_counter(); plan=fftw.fftw_plan_dft_r2c_1d(n,x.ctypes.data,spectrum.ctypes.data,64) # FFTW_ESTIMATE
for _ in range(s['fft_batch']): fftw.fftw_execute(plan)
```
The result cell below shows the spectrum, timings and verified agreement. Find the peaks before reading the timing comparison.


In [ ]:
fft = show_experiment('fft')
if fft:
    lesson_table(['Samples N','Manual DFT (ms)','FFTW execution (ms)','Execution speedup','Planning (ms)','Peaks'],
                 [[r['n'],f"{r['dft']['median_s']*1000:.3f}",f"{r['fftw']['median_s']*1000:.6f}",
                   f"{r['dft']['median_s']/r['fftw']['median_s']:.1f}×",f"{r['planning_s']*1000:.3f}",r['strongest_bins']] for r in fft['rows']])
    import numpy as np
    sample=fft['rows'][0];data=np.load(RUN_DIR/f"spectrum_{sample['n']}.npz")
    def draw(plt):
        fig,axes=plt.subplots(1,2,figsize=(10,3.5))
        axes[0].plot(data['signal']);axes[0].set(title='Signal: two frequencies mixed together',xlabel='Sample',ylabel='Sensor value')
        axes[1].stem(np.arange(min(16,len(data['spectrum']))),np.abs(data['spectrum'][:16]))
        axes[1].set(title='FFTW separates the frequencies',xlabel='Frequency bin (cycles in window)',ylabel='Amplitude')
        fig.tight_layout()
    lesson_plot(draw)
    lesson_text('**Answer:** the spectrum peaks at 3 and 7. The result checks passed for all measured sizes. The speedup compares transform execution; planning is a separate setup cost.')
    lesson_details('Library and complete measurements',fft);lesson_code('fft_lab')


## 2 — Which software actually did the work?
**Situation:** your notebook kernel and the compute job can use different Python packages and library paths.

**What actually runs:** the batch script loads the compiler/MPI toolchain and compatible libraries. The worker records what it actually loaded. This activity displays that evidence; `module load` selects installed software rather than installing it.

**Look for:** the OpenBLAS and FFTW paths, HDF5 backend, Python version and host. A library name on a slide is not evidence that it ran. This notebook reports actual calls and checks.

**Try:** inspect the expanded module list and find the compiler/MPI/HDF5 toolchain versions. Explain why libraries built for incompatible environments can fail to load.

**Learn:** record the environment with the results, so someone else can repeat the experiment.


In [ ]:
if REPORT:
    manifest=json.loads((RUN_DIR/'manifest.json').read_text())
    lesson_table(['Evidence','Actually used'],[['Compute host',manifest['system']['host']],['Python',manifest['software']['python'].splitlines()[0]],['NumPy',manifest['software']['numpy']],['BLAS',REPORT['experiments']['blas'].get('library','Unavailable')],['FFTW',REPORT['experiments']['fft'].get('fftw_library','Unavailable')],['HDF5',REPORT['experiments']['hdf5'].get('backend','Unavailable')]])
    lesson_details('Loaded modules and allocation',{'modules':manifest['software']['modules'],'resources':manifest['resources']})
else: print('NOT RUN: no environment evidence')


## 2A — BLAS: manual matrix multiplication versus a library
**Problem:** apply many sensor/calibration transformations as a dense matrix product `C = A @ B`. Same seeded inputs and double precision for all three implementations:
- `manual_ijk`: compiled C, simple row/column/dot-product loops.
- `manual_ikj`: compiled C, reorders loops so the inner loop accesses contiguous elements.
- `cblas_dgemm`: an actual BLAS call, initially **one thread**. Each full output is checked with a floating-point tolerance.

Warmup, repeated measurements and rotating method order reduce timing bias. Allocation, random generation and correctness checks are outside timing. Both C references use `-O3 -march=native`; Python only coordinates calls. Report both manual baselines so cache locality is visible alongside library optimization. A generic system BLAS may be slower than OpenBLAS: measure your implementation.

**Predict → change → explain:** (1) Double one size in `blas_sizes`: cubic work suggests about 8× operations, but cache effects change timing. (2) Change only `blas_threads` from 1 to 2 and rerun setup onwards; Slurm requests 2 CPUs per task. The report keeps the one-thread comparison and adds a two-thread BLAS timing. Small matrices can get slower. This second experiment requires OpenBLAS runtime thread controls and enough allocated CPUs. Configured threads do not prove every worker was active.

**Read the call:** `cblas_dgemm(101,111,111,n,n,n,1,A,n,B,n,0,C,n)` means row-major, no transposes, square dimensions, alpha=1, beta=0. BLAS handles arithmetic; you supply correct dimensions/storage. No explicit `#pragma omp parallel` is needed in the caller when the library manages its own threads.

Reference: [OpenBLAS runtime API](https://www.openmathlib.org/OpenBLAS/docs/extensions/).

### The operations executed
**Basic manual product — compiled C:**

```c
for(int i=0;i<n;i++) for(int j=0;j<n;j++) {
    double sum=0;
    for(int k=0;k<n;k++) sum+=a[i*n+k]*b[k*n+j];
    c[i*n+j]=sum;
}
```
**Reordered manual product — compiled C:**

```c
memset(c,0,(size_t)n*n*sizeof(double));
for(int i=0;i<n;i++) for(int k=0;k<n;k++) {
    double v=a[i*n+k];
    for(int j=0;j<n;j++) c[i*n+j]+=v*b[k*n+j];
}
```
**Actual library operation:** Python calls the compiled CBLAS routine. `f` is the bound DGEMM function; pointers identify the same arrays used by the manual routines.

```python
f=lib.cblas_dgemm
f(
    101, 111, 111, n, n, n, 1.0, a.ctypes.data, n, b.ctypes.data, n, 0.0,
    c.ctypes.data, n
)
```
`101` selects row-major storage, `111` means no transpose, alpha is 1 and beta is 0: **C = A × B**. The table below compares these three executed implementations.


In [ ]:
blas = show_experiment('blas')
if blas:
    from IPython.display import display, Markdown, HTML
    import html
    rows = blas['rows']
    table = [
        '| Matrix | Basic C loop (ms) | Reordered C loop (ms) | BLAS, 1 thread (ms) | BLAS vs basic | BLAS vs reordered |',
        '|---|---:|---:|---:|---:|---:|',
    ]
    for r in rows:
        t = r['timings']
        table.append(f"| {r['n']} × {r['n']} | {t['manual_ijk']['median_s']*1000:.3f} | "
                     f"{t['manual_ikj']['median_s']*1000:.3f} | {t['blas_1']['median_s']*1000:.3f} | "
                     f"{r['blas_speedup_over_ijk']:.2f}× | {r['blas_speedup_over_ikj']:.2f}× |")
    display(Markdown('**Same matrix product, three implementations. Lower time is better.**\n\n' + '\n'.join(table)))
    largest = max(rows, key=lambda r:r['n'])
    t = largest['timings']
    basic = t['manual_ijk']['median_s']; reordered = t['manual_ikj']['median_s']; optimized = t['blas_1']['median_s']
    change = 100*(1-optimized/reordered)
    time_change = f'{abs(change):.1f}% '+('less time' if change>=0 else 'more time')
    display(Markdown(
        f"**How to explain the largest matrix ({largest['n']} × {largest['n']}):**\n\n"
        f"- **Change the loop order:** {basic*1000:.3f} → {reordered*1000:.3f} ms, "
        f"**{basic/reordered:.2f}× the basic-loop speed**, using contiguous memory access.\n"
        f"- **Then use BLAS:** {reordered*1000:.3f} → {optimized*1000:.3f} ms, "
        f"**{reordered/optimized:.2f}× the reordered-loop speed**. "
        f"That is **{time_change}**.\n"
        f"- **Overall:** BLAS is **{basic/optimized:.2f}× the basic-loop speed**; "
        f"all three outputs passed the numerical checks. The baseline matters: these ratios compare different starting points.\n\n"
        "This comparison uses **one BLAS thread**. The gain comes from library optimization, not extra threads. "
        "The library may use SIMD within one core. Timings describe this run; a different BLAS implementation or machine can give different results."
    ))
    try:
        import matplotlib.pyplot as plt
        methods = [('manual_ijk','Basic C loop'),('manual_ikj','Reordered C loop'),('blas_1','BLAS — 1 thread')]
        fig, ax = plt.subplots(figsize=(9,4))
        positions = list(range(len(rows))); width = 0.25
        for j,(key,label) in enumerate(methods):
            values = [r['timings'][key]['median_s']*1000 for r in rows]
            bars = ax.bar([x+(j-1)*width for x in positions],values,width,label=label)
            ax.bar_label(bars,labels=[f'{v:.3f}' for v in values],padding=3,fontsize=8)
        ax.set_xticks(positions,[f"{r['n']} × {r['n']}" for r in rows])
        ax.set_yscale('log'); ax.set_ylabel('Median time (ms, logarithmic scale) — lower is better')
        ax.set_title('Identical matrix multiplication: memory access + BLAS optimization')
        ax.set_ylim(top=max(r['timings'][key]['median_s']*1000 for r in rows for key,_ in methods)*4)
        ax.legend();ax.grid(axis='y',alpha=0.2);ax.set_axisbelow(True)
        fig.tight_layout();plt.show()
    except ImportError:
        print('Chart optional: matplotlib unavailable; the measured table is complete.')
    threaded = [r for r in rows if 'blas_requested' in r['timings']]
    if threaded:
        table = ['| Matrix | BLAS 1 thread (ms) | BLAS requested threads (ms) | Configured threads | Thread speedup |',
                 '|---|---:|---:|---:|---:|']
        for r in threaded:
            one = r['timings']['blas_1']['median_s']; many = r['timings']['blas_requested']['median_s']
            table.append(f"| {r['n']} × {r['n']} | {one*1000:.3f} | {many*1000:.3f} | {r['configured_threads']} | {one/many:.2f}× |")
        display(Markdown('**Separate experiment: do extra BLAS threads help?**\n\n'+'\n'.join(table)+
                         '\n\nAbove 1× means faster; below 1× means slower. Configured threads do not measure actual worker utilization.'))
    else:
        display(Markdown('**Next student experiment:** change only `blas_threads` from 1 to 2, rerun setup onwards, and compare the new thread-speedup table.'))
    details = 'Library: '+blas['library']+'\nBuild: '+blas['configuration']+'\n\nRaw measurements and checks:\n'+json.dumps(rows,indent=2)+'\n\nCompiled C references:\n'+SOURCES['m3s4_linalg.c']
    display(HTML('<details><summary>Inspect C code, library details and raw measurements</summary><pre>'+html.escape(details)+'</pre></details>'))


## 2B — Calculate the temperature along a heated rod
**Situation:** a rod has one end at `rod_left`, the other at `rod_right`, and heating inside (`rod_source`). We want the temperature at every interior grid point.

**What actually runs:** form the heat-balance equations `A x = b`. The unknown vector x contains the temperatures. Solve them using (1) a compiled C teaching solver and (2) real LAPACK DGESV. Both use row swapping when necessary. Fresh input copies are supplied because both solvers overwrite their work arrays.

**Look for:** a plausible temperature profile, tiny equation residuals, and agreement with the known analytic solution. Residual means “how far do the computed temperatures miss the equations?” Timing is useful only after these checks pass.

**A second real test:** the small system `[[0,1],[1,1]] x = [1,2]` has answer `[1,1]`. A solver that refuses to swap rows stops at its first zero pivot. Both pivoting solvers succeed. A separate singular system returns `INFO>0`, telling us not to accept a unique solution.

**Try:** change only `rod_source` to 0 and rerun. Predict a straight line between the two end temperatures. Then change heating or one end temperature in another run; explain the plot.

**Learn:** LAPACK supplies a tested solver, but you must still give it the right data layout and inspect its status. The C reference uses row-major A; the Fortran interface receives column-major A and 32-bit LP64 integer arguments. A production rod solver should exploit the tridiagonal structure instead of treating this as a general dense system.

### The operations executed
**Build the heat-balance system:**

```python
dx=1/(n+1);a=2*np.eye(n)-np.eye(n,k=1)-np.eye(n,k=-1)
b=np.full(n,source*dx*dx);b[0]+=left;b[-1]+=right
```
**Solve it using LAPACK:** `solve` is the real DGESV routine. Column-major `aa` contains A, `bb` contains b and receives the solution, `piv` receives row swaps, and `info` reports success/failure.

```python
solve=lib.dgesv_
solve(
    ct.byref(n), ct.byref(one), aa.ctypes.data, ct.byref(n), piv.ctypes.data,
    bb.ctypes.data, ct.byref(n), ct.byref(info)
)
```
The table checks both solvers against the equations and the analytic temperature profile; the plot displays the solution returned by LAPACK.


In [ ]:
lapack=show_experiment('lapack')
if lapack:
    lesson_table(['Solver','Median solve time (ms)','Relative residual','Maximum analytic error'],
                 [[label,f"{lapack['timings'][key]['median_s']*1000:.3f}",f"{lapack['checks'][key]['relative_residual']:.2e}",f"{lapack['checks'][key]['max_analytic_error']:.2e}"] for key,label in [('manual_pivot','Manual C with pivoting'),('lapack','LAPACK DGESV')]])
    pivot=lapack['pivot_example']
    lesson_table(['Test','Observed result','Meaning'],[['Ordinary heat system','Both checks passed','Computed temperatures satisfy the heat balance'],['Zero first pivot',f"Without row swapping: stops (code {pivot['no_pivot_info']}); LAPACK INFO={pivot['lapack_info']}",'Swap rows to solve the nonsingular system'],['Singular matrix',f"LAPACK INFO={lapack['singular_info']}",'No unique solution: do not accept the output as a solution']])
    def draw(plt):
        plt.figure(figsize=(7,3.5));plt.plot([0]+lapack['positions']+[1],[lapack['boundaries'][0]]+lapack['temperatures']+[lapack['boundaries'][1]],label='Computed temperature')
        plt.plot([0,1],lapack['boundaries'],'--',label='Straight line without heating')
        plt.xlabel('Position along unit-length rod');plt.ylabel('Temperature');plt.title(f"Interior heating = {lapack['source']}");plt.legend();plt.tight_layout()
    lesson_plot(draw)
    lesson_text('**Interpret:** the solid curve gives the calculated temperatures. Setting `rod_source=0` should make it follow the dashed line. Small residuals check the answer; a faster runtime alone does not.')
    lesson_details('Complete LAPACK evidence',lapack);lesson_code('lapack_lab')


## 2D — Save progress, stop the program, then continue
**Situation:** a long simulation may stop before finishing. A checkpoint is a saved game: it stores the current state so we can continue from there.

Our small simulation models a rod with a warm centre and cold ends. Each step updates the heat values using neighboring values. Here “temperature” is a relative value, not a calibrated Kelvin measurement.

**What actually runs — three separate processes:**
1. **Reference:** start at step 0 and finish all `heat_steps`, without interruption.
2. **Failure experiment:** start again, save the current grid every `checkpoint_interval` steps, and deliberately terminate at `failure_step`. This is an intentional process failure, not a kernel crash.
3. **Restart:** read the last checkpoint from disk and continue to the final step. Compare its full final array with the uninterrupted reference; they must match exactly.

**Predict with the defaults:** save every 10 steps and fail at step 57. The latest save is step 50. Steps 51–56 are completed work that must be repeated; step 57 is retried too. The display below distinguishes those six completed steps from the failing step.

**Try:** change only `checkpoint_interval` to 5, then 25, keeping failure at 57. Rerun the job each time. Before running, predict the last saved step, lost completed steps and number of saves.

**Learn:** frequent saves can reduce repeated computation but add saving overhead. Saving only the random seed cannot restore the current grid. This is application-state checkpointing; it does not capture an entire MPI job or guarantee recovery after a power failure.

### The operations executed
**Advance one heat step:**

```python
def heat_step(a):
    out = a.copy()
    out[1:-1] = a[1:-1] + 0.2 * (a[:-2] - 2 * a[1:-1] + a[2:])
    return out
```
Each interior cell uses its two neighbors. The loop repeats this update for each simulation step.

**Save the current state and deliberately stop the failing process:**

```python
if mode == 'fail' and step % s['checkpoint_interval'] == 0:
    save(cp, {
        'schema': 1,
        'step': step,
        'parameters': identity,
        'grid': a.tolist(),
        'grid_sha256': hashlib.sha256(a.tobytes()).hexdigest()
    })
```

```python
if mode == 'fail' and step == s['failure_step']:
    os._exit(17)
```
These are separate operation excerpts. In the actual loop, the failure check runs before saving: a failure on a checkpoint step occurs before that step is saved. The full worker also records failure statistics. Before restarting, it reads the saved grid and step:

```python
state=json.loads(cp.read_text()); assert state['schema']==1 and state['parameters']==identity
a=np.array(state['grid'],dtype=float); start=state['step']
```
**Start a separate restart process and check its answer:**

```python
checked(cmd+['resume'],root)
```

```python
actual=np.load(restarted)
assert np.array_equal(ref,actual)
```
`cmd` launches the worker with `--heat-child`; mode `resume` reads the checkpoint. The stages, lost work and final-state comparison appear below. The complete worker includes integrity checks and saving-time measurements.


In [ ]:
checkpoint=show_experiment('checkpoint')
if checkpoint:
    saved=checkpoint['resumed_from_step'];failure=checkpoint['failure_step'];lost=checkpoint['completed_steps_lost'];settings=REPORT['settings'];total=settings['heat_steps']
    lesson_table(['Stage','What really happened'],[['1. Uninterrupted reference',f"A separate process computed steps 1–{total} and saved its final array."],['2. Save and fail',f"Another process saved every {settings['checkpoint_interval']} steps, then terminated intentionally at step {failure}."],['3. Restart',f"Loaded saved step {saved}; continued at step {saved+1}." if saved else 'No checkpoint existed; restarted from initial conditions.'],['4. Verify','The complete restarted final array equals the uninterrupted final array exactly.']])
    lesson_table(['Checkpoint evidence','Observed value','Interpretation'],[['Last saved step',saved,'Progress available after failure'],['Lost completed steps',lost,f"Completed steps {saved+1}–{failure-1} must be repeated" if lost else 'No completed steps since the saved state'],['Failing step',failure,'Retried as well; not counted among lost completed steps'],['Checkpoint writes',checkpoint['checkpoint_writes'],'Number of saves before failure'],['Saved file size',f"{checkpoint['checkpoint_bytes']} bytes",'Grid plus step, parameters and integrity hash'],['Time spent saving',f"{checkpoint['checkpoint_seconds']*1000:.3f} ms",'Measured saving cost before failure']])
    lesson_text(f"**Result:** resume from **step {saved}**, rather than start over. Repeat **{lost} previously completed steps**, retry the failing step, and reach the **same final answer**.")
    import numpy as np
    ref=np.load(RUN_DIR/'reference_final.npy');resume_path=RUN_DIR/'resume_final.npy';restarted=np.load(resume_path) if resume_path.exists() else ref
    state=json.loads((RUN_DIR/'checkpoint.json').read_text()) if (RUN_DIR/'checkpoint.json').exists() else None
    def draw(plt):
        fig,axes=plt.subplots(1,2,figsize=(11,3.7));positions=np.linspace(0,1,len(ref))
        axes[0].plot(positions,np.sin(np.pi*positions),'--',label='Initial state (step 0)')
        if state:axes[0].plot(positions,state['grid'],label=f'Saved state (step {saved})')
        axes[0].plot(positions,ref,label=f'Uninterrupted final (step {total})')
        axes[0].plot(positions,restarted,':',label='Restarted final — overlaps reference')
        axes[0].set(title='The checkpoint saves this entire heat field',xlabel='Position along rod',ylabel='Relative temperature');axes[0].legend(fontsize=8)
        if saved:axes[1].barh('Before failure',saved,left=0,color='seagreen',label='Saved progress')
        if lost:axes[1].barh('Before failure',lost,left=saved,color='darkorange',label='Lost completed work')
        axes[1].barh('After restart',total-saved,left=saved,color='steelblue',label='Recomputed / remaining work')
        axes[1].axvline(failure,color='firebrick',linestyle='--',label=f'Failure at {failure}')
        axes[1].set(xlim=(0,total),xlabel='Simulation step',title=f'Restart begins after saved step {saved}');axes[1].legend(fontsize=8)
        fig.tight_layout()
    lesson_plot(draw)
    predictions=[]
    for interval in [5,10,25]:
        last=(failure-1)//interval*interval
        predictions.append([interval,last,failure-1-last,(failure-1)//interval])
    lesson_text('**Predict your next runs — these rows are calculations, not extra simulations:**')
    lesson_table(['Save interval','Predicted last saved step','Predicted lost completed steps','Predicted saves before failure'],predictions)
    lesson_text('Change only `checkpoint_interval`, rerun the job, and check the prediction against measured saving time. More frequent saving trades less repeated work for more writes.')
    lesson_code('heat_step','heat_child','checkpoint_lab');lesson_details('Raw checkpoint evidence',checkpoint)


## 3 — A faster calculation can reveal the next bottleneck
**Situation:** a weather program spends 8 seconds computing and 2 seconds writing results.

**What executes here:** a small arithmetic model in the notebook, not a measured weather simulation. Change `compute_s`, `io_s` or `compute_speedup` directly in this cell; rerun just this cell.

**Predict:** if computation becomes 4× faster, will the whole program also become 4× faster?

**Look for:** computing falls from 8 to 2 seconds, but writing stays at 2. Total time falls from 10 to 4 seconds: only 2.5× faster. I/O now accounts for half the runtime.

**Learn:** optimize the whole application. Ask what limits performance after each successful improvement.


In [ ]:
compute_s, io_s, compute_speedup = 8.0, 2.0, 4.0
assert compute_s>0 and io_s>=0 and compute_speedup>0
before=compute_s+io_s;after=compute_s/compute_speedup+io_s
lesson_table(['MODEL — not measured','Computation (s)','Writing (s)','Total (s)'],[['Before',compute_s,io_s,before],['After',compute_s/compute_speedup,io_s,after]])
lesson_text(f"**Whole-program speedup: {before/after:.2f}×.** Writing grows from {100*io_s/before:.1f}% to {100*io_s/after:.1f}% of runtime. Explain why it can become the next bottleneck even though it did not become slower.")


## 4 — Write the same data as many files or one file
**Situation:** a simulation produces many records. Should each record become a separate file?

**What actually runs:** one writer creates `files` individual files, then one file containing the same records in the same order. Repeated trials rotate the order. The worker verifies equal total bytes and the complete content checksum. Directory creation and verification are outside timing; file creation, writing and close are timed.

**Predict:** which layout will take longer, although both write exactly the same data?

**Try:** change `files` from 200 to 400 and `bytes_per_file` from 4096 to 2048 in a separate exploration to keep total bytes fixed. This changes two controls, so do not use that pair for the one-setting exercise in §10. Alternatively change only `fsync` and discuss durability cost.

**Learn:** opening and managing files has a cost. Results depend on the measured filesystem and warm cache; this is not a measurement of physical disk bandwidth or parallel I/O.

### The operations executed
`payloads` contains the same records in both cases. `d` is a temporary directory inside this run.

**Many files: opening is inside the record loop.**

```python
for i,payload in enumerate(payloads):
    with open(d/f'{i:06d}.bin','wb') as f:
        f.write(payload)
```
**One file: open once, then write all records.**

```python
with open(d/'all.bin','wb') as f:
    for payload in payloads: f.write(payload)
```
The `with` block closes the file. With `fsync=True`, the full worker also flushes and synchronizes each file. These are core excerpts; repeated timing and content verification remain in the complete worker.

**What the result measures:** the time to perform these file operations for equal total bytes. More `open`/`close` operations can add overhead. Look at the measured difference below, then explain it using the position of `open` in these two snippets.


In [ ]:
io=show_experiment('io')
if io:
    many=io['many']['median_s'];one=io['one']['median_s']
    lesson_table(['Layout','Files per trial','Total payload bytes','Median write/close (ms)'],[['Many files',REPORT['settings']['files'],io['bytes'],f'{many*1000:.3f}'],['One file',1,io['bytes'],f'{one*1000:.3f}']])
    lesson_text(f"**One-file throughput relative to many-file throughput: {many/one:.2f}×** for the same payload (above 1× is faster). Full content hashes matched. `fsync={io['fsync']}`. This measures one writer on the recorded filesystem, with a warm cache.")
    def draw(plt):
        fig,ax=plt.subplots(figsize=(6,3));bars=ax.bar(['Many files','One file'],[many*1000,one*1000]);ax.bar_label(bars,fmt='%.3f');ax.set_ylabel('Median write/close time (ms)');ax.set_title('Same data: different file-management cost');ax.set_ylim(top=max(many,one)*1250);fig.tight_layout()
    lesson_plot(draw);lesson_code('io_lab');lesson_details('All I/O repetitions and checks',io)


## 5 — Where did our bytes go?
**What executes here:** display the compute job’s filesystem and run directory. This does not run another storage benchmark.

**Look for:** NFS, a local filesystem, or another reported type. Do not assume `/home` is local storage, and do not call an NFS result a Lustre benchmark.

**Try:** explain why a result measured on shared home storage may differ from a job writing to scratch. To test another filesystem, change WORKSPACE to an authorized shared path visible to compute nodes and rerun.

**Learn:** a timing belongs to a specific path, machine and allocation. Keep that context with your results.

### The operation executed
The worker records the filesystem for the actual run directory:

```python
df=subprocess.run(['df','-T',str(root)],capture_output=True,text=True) if shutil.which('df') else None
```
The display below reads that saved record. Inspect the filesystem type before interpreting any I/O timing.


In [ ]:
if REPORT:
    lesson_table(['Measurement context','Recorded evidence'],[['Path',str(RUN_DIR)],['Compute host',manifest['system']['host']]])
    lesson_details('Filesystem type — inspect the Type column',manifest['system']['filesystem'] or 'No filesystem record')
    lesson_details('Job resource allocation',manifest['resources'])
else:print('NOT RUN: no storage evidence')


## 6 — Save weather data with structure, then read just one time
**Situation:** a weather calculation produces a temperature grid for several times. We want to save values together with their meaning.

**What actually runs:** the worker creates a synthetic `time × y × x` temperature field, writes an HDF5 `temperature` dataset with Kelvin units and a time axis, closes the file, reopens it, checks every value, and reads the selected `weather_slice`. Both Python and compiled C backends support the settings.

**Look for:** the selected time slice, dimensions, units, compression and file size. The selected slice should differ from neighboring times because the input formula increases temperature by 1 K per time index. This is a synthetic dataset, not a weather forecast.

**Try:** change only `weather_slice` to 1; rerun the job and verify the plotted slice. Then change only `compression` to `'gzip'`; compare bytes and write time against your first run. Compression can save space while adding work.

**Learn:** structured files preserve arrays and metadata, and a hyperslab selects a region without requesting the whole dataset in that read call. The timing here follows full readback, so the cache is warm. This is one writer; using h5pcc does not make it parallel. NetCDF/CF conventions add standardized climate dimensions and metadata conventions beyond this HDF5 example.

### The operations executed
**Actual C HDF5 calls used by the SciTech fallback** (handles and error-checking setup omitted):

```c
hid_t d=H5Dcreate2(f,"temperature",H5T_IEEE_F64LE,s,H5P_DEFAULT,props,H5P_DEFAULT);HANDLE(d);
CALL(H5Dwrite(d,H5T_NATIVE_DOUBLE,H5S_ALL,H5S_ALL,H5P_DEFAULT,field));
CALL(H5Sselect_hyperslab(s,H5S_SELECT_SET,offset,NULL,count,NULL));
CALL(H5Dread(d,H5T_NATIVE_DOUBLE,memory,s,H5P_DEFAULT,slice));double read_s=now()-start;
```
`field` is the temperature grid. `offset`/`count` select one time slice; `slice` receives it. `CALL` checks HDF5 errors. The report states which backend executed.

**Equivalent operations in the h5py backend:**

```python
ds=f.create_dataset('temperature',data=data,chunks=(1,shape[1],shape[2]),compression=s['compression'])
start=time.perf_counter(); region=ds[s['weather_slice'],:,:]; read_s=time.perf_counter()-start
```
The table and plot below show dimensions, bytes and the selected plane. The plot labels whether it reads the file directly or displays the input formula verified by the C program.


In [ ]:
hdf5=show_experiment('hdf5')
if hdf5:
    import numpy as np
    s=REPORT['settings'];shape=s['weather_shape'];selected=s['weather_slice'];result=hdf5.get('result',{})
    write_time=hdf5.get('write_close_s',result.get('write_close_s'));read_time=hdf5.get('read_slice_s',result.get('read_slice_s'))
    lesson_table(['Stored / checked item','Observed value'],[['Backend',hdf5['backend']],['Array shape: time × y × x',shape],['Temperature units','K'],['Selected time index',selected],['Compression',s['compression'] or 'None'],['Actual file bytes',hdf5['bytes']],['Write and close',f'{write_time*1000:.3f} ms'],['Selected-slice read',f'{read_time*1000:.3f} ms (warm cache)'],['Verification','Full field and selected slice matched the original data']])
    # The C fallback keeps its small readback internal. Make the plot's source
    # explicit: plot actual file values when h5py is available, otherwise the
    # input formula whose complete stored values the C program verified.
    try:
        import h5py
        with h5py.File(RUN_DIR/'weather.h5','r') as f:plane=f['temperature'][selected,:,:]
        origin='Read from the actual weather.h5 file'
    except ImportError:
        y,x=np.indices(shape[1:]);plane=273.15+selected+0.1*y+0.01*x
        origin='Expected slice from input formula; compiled C verified the stored slice'
    def draw(plt):
        fig,ax=plt.subplots(figsize=(6,3.5));im=ax.imshow(plane,origin='lower',aspect='auto');fig.colorbar(im,ax=ax,label='Temperature (K)');ax.set(title=f'Time slice {selected}',xlabel='x grid index',ylabel='y grid index');fig.tight_layout()
    lesson_plot(draw);lesson_text('**Plot source:** '+origin+'. Change `weather_slice`; predict how its temperatures change before rerunning.')
    lesson_code('hdf5_lab');lesson_details('C HDF5 source',SOURCES['m3s4_hdf5.c']);lesson_details('Complete HDF5 evidence',hdf5)


## 7 — Four real processes write one shared file
**Situation:** parallel workers each produce a different block of output. They must write to distinct locations rather than overwrite each other.

**What actually runs:** `srun` starts real MPI processes. Rank r owns the bytes starting at `r × mpi_block`. All ranks call `MPI_File_write_at_all`, synchronize/close, reopen, and read their own block. Each byte, file size and global success are checked; Python independently checks the complete file.

**Look for:** the ownership table below. With four ranks and 1024 bytes each, the complete file is 4096 bytes. This is the notebook’s explicit process parallelism.

**Try:** change only `mpi_ranks` to 2; rerun the job and predict the new file size and byte ranges. Then return to four ranks and change only `mpi_block`.

**Learn:** collective means all ranks participate; it does not mean they all write to offset zero. A correctness check on one node does not prove multi-node storage scaling.

### The operations executed
**Actual compiled MPI code** (MPI setup omitted; `CHECK` verifies return codes):

```c
CHECK(MPI_File_write_at_all(fh,(MPI_Offset)rank*block,
                            buf,block,MPI_BYTE,&status));
CHECK(MPI_File_read_at_all(fh,(MPI_Offset)rank*block,
                           buf,block,MPI_BYTE,&status));
```
The byte offset is **rank × block**. All MPI processes execute these collective calls, each with its own offset and data. The table below maps that expression to the actual byte ranges; the saved launch command confirms how many processes ran.


In [ ]:
mpiio=show_experiment('mpiio')
if mpiio:
    r=mpiio['result'];block=r['block']
    lesson_table(['MPI rank','Bytes owned','Offset passed to collective write'],[[rank,f'{rank*block} through {(rank+1)*block-1}',rank*block] for rank in range(r['ranks'])])
    lesson_text(f"**Executed {r['ranks']} real processes.** Each wrote {block} bytes; the checked file contains **{r['bytes']} bytes**. All ranks participated in the collective calls and every byte passed readback checks. Maximum write/sync/close time: **{r['max_write_sync_close_s']*1000:.3f} ms**.")
    lesson_details('Actual process launch command',mpiio['command']);lesson_details('Actual MPI C code — find write_at_all and rank × block',SOURCES['m3s4_mpiio.c']);lesson_details('Complete MPI evidence',mpiio)


## 8 — Same calculation, controlled and uncontrolled randomness
**Situation:** two runs of a noisy calculation give different answers. Is the code wrong, or did the experiment change?

**What actually runs:** the worker creates an input sensor dataset using `input_seed`, adds noise using `seed`, and computes the mean squared value. It also records two unseeded runs and controlled changes to the noise seed and magnitude.

**Look for:** the unchanged seed reproduces the numerical result, while another seed or noise size can change it. Small changes are expected in this experiment.

**Try:** set `noise_sigma=0`; rerun. Predict whether changing the noise seed should still matter. It should not: zero noise removes that source of randomness.

**Learn:** a seed controls randomness, not the input data, code version, software or timing. Reproducibility needs those too.

### The operations executed
The actual sensor statistic is a short NumPy calculation:

```python
def simulation(data,seed,sigma):
    noise=np.random.default_rng(seed).normal(0,sigma,size=data.shape)
    return float(np.mean((data+noise)**2))
```
A fresh generator with the recorded seed produces the same noise for the same input. The table below compares controlled and uncontrolled changes.


In [ ]:
if REPORT:
    replay=REPORT['experiments']['replay'];values=REPORT['unseeded_results']
    lesson_table(['Experiment','Mean squared sensor value','What changed'],[['Unseeded run A',f'{values[0]:.12f}','Noise seed not controlled'],['Unseeded run B',f'{values[1]:.12f}','Noise seed not controlled'],['Recorded run',f"{replay['recorded']:.12f}",'Recorded input and noise seeds'],['Fresh replay',f"{replay['replayed']:.12f}",'Nothing: check repeatability'],['Changed noise seed',f"{replay['changed_seed_result']:.12f}",'Seed only'],['Doubled noise magnitude',f"{replay['changed_sigma_result']:.12f}",'Noise sigma only']])
    lesson_text('The unseeded values can differ because their noise differs. The replay agrees because the input, source and parameters were checked. With sigma zero, noise-seed changes should have no effect.')
    lesson_code('simulation','replay')
else: print('NOT RUN: reproducibility activity')


## 9 — Can another process repeat the recorded calculation?
**What actually runs in this cell:** start a fresh Python process using the saved manifest. It checks the runner source and input hashes, loads recorded parameters, recomputes the noisy sensor statistic and compares with the saved result.

**Look for:** the recorded and replayed numbers agree within tolerance. A numerical match does not require identical runtime.

**Try next:** the following cell damages a separate input copy and launches replay against that copy. Predict whether it should calculate a new answer or reject the input first.

**Learn:** keep a usable recipe and check data integrity before calculating. A manifest is more useful than a screenshot of one answer.

### The operations executed
These statements run in the **fresh replay process**:

```python
inp=root/m['input']['path']; assert sha(inp)==m['input']['sha256'], 'input checksum mismatch'
params=m['parameters']; result=simulation(np.load(inp),params['seed'],params['noise_sigma'])
assert math.isclose(result,m['result']['value'],rel_tol=1e-12,abs_tol=1e-12), 'scientific result differs'
```
The input hash must match before calculation. The recorded parameters are used to recompute the statistic, then its value is compared within tolerance. The cell below starts this process and shows its returned results.


In [ ]:
if REPORT:
    cmd = [sys.executable, str(RUN_DIR / 'm3s4_lab.py'), '--replay', str(RUN_DIR / 'manifest.json')]
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
    if p.returncode: raise RuntimeError(p.stderr)
    result=json.loads(p.stdout)
    lesson_table(['Replay check','Observed value'],[['Recorded statistic',result['recorded']],['Fresh-process statistic',result['replayed']],['Status',result['status']]])
    lesson_text('The recorded calculation was executed again in a separate process after source/input integrity checks.')
    print('Manifest:', RUN_DIR / 'manifest.json')
else: print('NOT RUN: replay')


### Does replay reject a changed input?
Make a separate damaged copy, never overwrite the real dataset. A controlled integrity failure should occur before the numerical calculation. This is a test of the evidence check, not an application crash to fix.


In [ ]:
if REPORT:
    tamper = RUN_DIR / 'tamper_test'
    tamper.mkdir(exist_ok=True)
    altered = bytearray((RUN_DIR / 'input.npy').read_bytes())
    altered[-1] ^= 1
    (tamper / 'input.npy').write_bytes(altered)
    (tamper / 'manifest.json').write_text((RUN_DIR / 'manifest.json').read_text())
    p = subprocess.run([sys.executable, str(RUN_DIR / 'm3s4_lab.py'), '--replay', str(tamper / 'manifest.json')],
                       capture_output=True, text=True, timeout=30)
    assert p.returncode != 0 and 'input checksum mismatch' in p.stderr
    print('PASS: altered input rejected before use; original evidence preserved.')
else: print('NOT RUN: integrity test')


## 10 — Your experiment: predict, change, compare, explain
1. Choose one activity and write a prediction, e.g. “saving every 5 steps will lose less work but write more checkpoints.”
2. Copy the current RUN_DIR into `BASELINE_DIR` below before changing settings.
3. Change **one SETTINGS value**, rerun setup/materialize/submit/collect, then rerun this comparison cell.
4. Read the relevant baseline/current table, decide whether your prediction held, and explain a trade-off.

**What executes here:** compare two saved reports. It does not rerun either experiment. Keep the same source files and execution environment when attributing an effect to one setting.

**Submit:** prediction, setting change, two run IDs, one table/plot, correctness evidence, and two sentences explaining the result. “PASS” alone is not the learning outcome.

### The operations executed
This activity compares two saved experiment reports; it does not rerun the scientific kernel:

```python
old=json.loads((baseline_path/'report.json').read_text())
changed=[k for k in SETTINGS if old['settings'].get(k)!=SETTINGS[k]]
assert len(changed)==1,'Change exactly one setting: '+str(changed)
```
The result table selects the measurements affected by that one setting. Explain those changes before interpreting unrelated timing variation.


In [ ]:
BASELINE_DIR = None  # Paste the first RUN_DIR here, e.g. '/home/odiez/m3s4_runs/run-...'
if REPORT and BASELINE_DIR:
    baseline_path=Path(BASELINE_DIR)
    old=json.loads((baseline_path/'report.json').read_text())
    changed=[k for k in SETTINGS if old['settings'].get(k)!=SETTINGS[k]]
    assert len(changed)==1,'Change exactly one setting: '+str(changed)
    old_manifest=json.loads((baseline_path/'manifest.json').read_text())
    current_manifest=json.loads((RUN_DIR/'manifest.json').read_text())
    assert old_manifest['sources']==current_manifest['sources'],'Source versions differ: use a baseline with the same experiment implementation'
    lesson_text(f"**Changed setting:** `{changed[0]}`: `{old['settings'][changed[0]]}` → `{SETTINGS[changed[0]]}`. Baseline `{old['run_id']}`; current `{REPORT['run_id']}`.")
    def measurements(report):
        values={}
        for name,r in report['experiments'].items():
            if r['status']!='PASS':continue
            if name=='blas':
                for row in r['rows']:
                    for method,t in row['timings'].items():values[f"BLAS N={row['n']} {method} (ms)"]=t['median_s']*1000
            elif name=='fft':
                for row in r['rows']:
                    for method in ['dft','fftw']:values[f"FFT N={row['n']} {method} (ms)"]=row[method]['median_s']*1000
            elif name=='checkpoint':
                for key in ['resumed_from_step','completed_steps_lost','checkpoint_writes','checkpoint_bytes']:values['Checkpoint '+key]=r[key]
                values['Checkpoint saving (ms)']=r['checkpoint_seconds']*1000
            elif name=='io':
                for layout in ['many','one']:values['I/O '+layout+' (ms)']=r[layout]['median_s']*1000
            elif name=='lapack':
                for method,t in r['timings'].items():values['LAPACK '+method+' (ms)']=t['median_s']*1000
                for method,c in r['checks'].items():values['LAPACK '+method+' residual']=c['relative_residual']
            elif name=='hdf5':values['HDF5 file bytes']=r['bytes']
            elif name=='mpiio':values['MPI file bytes']=r['result']['bytes']
        return values
    first=measurements(old);second=measurements(REPORT)
    # Show the activity affected by the chosen setting instead of every dictionary.
    groups={'blas_sizes':'BLAS','blas_threads':'BLAS','sizes':'FFT','fft_batch':'FFT',
            'checkpoint_interval':'Checkpoint','failure_step':'Checkpoint','heat_steps':'Checkpoint','heat_points':'Checkpoint',
            'files':'I/O','bytes_per_file':'I/O','fsync':'I/O','lapack_points':'LAPACK','rod_left':'LAPACK','rod_right':'LAPACK','rod_source':'LAPACK',
            'weather_shape':'HDF5','weather_slice':'HDF5','compression':'HDF5','mpi_ranks':'MPI','mpi_block':'MPI'}
    group=groups.get(changed[0]);keys=sorted(k for k in set(first)|set(second) if group is None or k.startswith(group))
    def fmt(value):return 'Not measured' if value is None else f'{value:.6g}'
    lesson_table(['Measured quantity','Baseline','After change'],[[k,fmt(first.get(k)),fmt(second.get(k))] for k in keys])
    lesson_table(['Activity','Baseline status','Current status'],[[k,old['experiments'].get(k,{}).get('status','Absent'),v['status']] for k,v in REPORT['experiments'].items()])
    if changed[0] in ['seed','input_seed','noise_sigma']:
        lesson_table(['Statistic','Baseline','After change'],[['Mean squared sensor value',old['experiments']['replay']['recorded'],REPORT['experiments']['replay']['recorded']]])
    lesson_text('**Your conclusion:** did the predicted result occur? Explain the gain or trade-off. Changed timings in unaffected activities can be run-to-run noise; do not attribute every difference to your setting.')
else:
    lesson_text('**Before changing settings:** copy the current RUN_DIR. After the changed run completes, paste that first directory into BASELINE_DIR above and rerun this comparison cell.')


## 11 — Discussion: package and automate the same experiment
**This section is a design discussion; it does not build a container or run a workflow engine.**

- **Native environment:** selected modules/packages provide the compiler and libraries you used.
- **Container:** packages an application environment; it still depends on compatible host MPI, drivers and hardware.
- **Workflow:** expresses stages such as prepare → calculate → save → verify → analyze, plus retries and dependencies.

**Student task:** sketch how you would automate the checkpoint experiment and retain its manifest. Which saved artifact lets you restart? Which test prevents an incorrect result from being accepted?

**Learn:** packaging, orchestration and numerical validation solve different parts of repeatable scientific work.


## 12 — Knowledge check
Answer before expanding:
1. Where is real parallelism in this notebook?
2. Why is FFT planning separated from execution?
3. Why can equal byte counts have different I/O cost?
4. What state does the heat simulation need to restart?
5. How do serial HDF5 and MPI-IO differ?
6. Which checks could fail even if output size is correct?
7. Why is a numerical replay PASS not proof of identical timing?

<details><summary>Answers</summary>

1. MPI launches separate ranks; collective calls write disjoint owned regions. FFTW here is serial.
2. A reused plan amortizes setup; end-to-end applications must account for setup too.
3. Metadata/open/close count, pattern, caching, sync and storage affect cost.
4. Current grid, step and compatible equation parameters, plus format/integrity checks.
5. HDF5 describes structured datasets/metadata; MPI-IO coordinates distributed byte access. Parallel HDF5 is a separate configuration.
6. Data order, contents, counts, shape, units or checksums could be wrong.
7. Performance also depends on hardware, environment, allocation and load; compare repeated measured samples.

</details>

7. What does BLAS optimize beyond changing the loop order? Why compare one thread first?
8. Why does LAPACK swap rows, overwrite inputs and return INFO?
9. Why does compiling with h5pcc not make a single-writer program parallel?


## 13 — What did you demonstrate?
The status table below summarizes actual execution. **PASS** means the stated checks passed; **SKIPPED** means it did not run; **FAIL** needs investigation. Discussion sections and the arithmetic model are labeled separately.

Before leaving, explain one measured improvement, one correctness check, one trade-off and where real parallelism occurred. Retain the run directory and manifest so your claim can be checked.


In [ ]:
if REPORT:
    meaning={'blas':'Manual C products vs actual CBLAS; full output agreement','lapack':'Heat solve, residual/analytic checks, pivot and singular cases','fft':'Manual C DFT vs actual FFTW; spectra checked','io':'Many files vs one; equal payload verified','checkpoint':'Separate failure/restart processes; final arrays identical','hdf5':'Structured weather output; full and slice readback','mpiio':'Actual MPI ranks; offsets and every byte verified','replay':'Source/input hashes and numerical replay checked'}
    lesson_table(['Activity','Status','What was checked'],[[name.upper(),r['status'],meaning.get(name,'') if r['status']=='PASS' else r.get('reason','')] for name,r in REPORT['experiments'].items()])
    lesson_text('**Discussion/model only:** §3 arithmetic bottleneck model; §11 containers/workflow design; NetCDF/CF context. These are not reported as executed library demos.')
    print('Run:',REPORT['run_id'],'| Build:',REPORT['build'],'| Failed:',REPORT['failed'],'| Skipped:',REPORT['skipped'])
    print('Evidence:',RUN_DIR/'manifest.json')
else:print('NOT RUN: no execution evidence')
